In [1]:
# ============================================================
# CELL 1: ROAD–FLOOD GROUNDING SETUP
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

warnings.filterwarnings("ignore")

print("=" * 72)
print("ROAD–FLOOD GROUNDING")
print("=" * 72)

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"

PROCESSED_DATA_DIR = DATA_DIR / "processed"

# Transportation knowledge generated in Notebook 05
TRANSPORTATION_KNOWLEDGE_DIR = (
    PROCESSED_DATA_DIR
    / "transportation_knowledge"
)

ROAD_KNOWLEDGE_DIR = (
    TRANSPORTATION_KNOWLEDGE_DIR
    / "road_knowledge"
)

MASTER_TRANSPORTATION_DIR = (
    TRANSPORTATION_KNOWLEDGE_DIR
    / "master"
)

SCENE_MANIFEST_PATH = (
    MASTER_TRANSPORTATION_DIR
    / "transportation_scene_manifest.csv"
)

MASTER_PROFILE_PATH = (
    MASTER_TRANSPORTATION_DIR
    / "master_scene_transportation_profiles.csv"
)

MASTER_TOKEN_PATH = (
    MASTER_TRANSPORTATION_DIR
    / "master_scene_transportation_tokens.csv"
)

# SEN1Floods11 acquisition outputs
SEN1FLOODS11_DIR = (
    PROCESSED_DATA_DIR
    / "sen1floods11"
)

# Candidate imagery folders
S1_DIR = (
    SEN1FLOODS11_DIR
    / "S1Hand"
)

S2_DIR = (
    SEN1FLOODS11_DIR
    / "S2Hand"
)

LABEL_DIR = (
    SEN1FLOODS11_DIR
    / "LabelHand"
)

JRC_WATER_DIR = (
    SEN1FLOODS11_DIR
    / "JRCWaterHand"
)

# Notebook 06 outputs
ROAD_FLOOD_OUTPUT_DIR = (
    PROCESSED_DATA_DIR
    / "road_flood_grounding"
)

ROAD_LEVEL_OUTPUT_DIR = (
    ROAD_FLOOD_OUTPUT_DIR
    / "road_level"
)

SCENE_LEVEL_OUTPUT_DIR = (
    ROAD_FLOOD_OUTPUT_DIR
    / "scene_level"
)

TOKEN_OUTPUT_DIR = (
    ROAD_FLOOD_OUTPUT_DIR
    / "tokens"
)

LOG_OUTPUT_DIR = (
    ROAD_FLOOD_OUTPUT_DIR
    / "logs"
)

for directory in [
    ROAD_FLOOD_OUTPUT_DIR,
    ROAD_LEVEL_OUTPUT_DIR,
    SCENE_LEVEL_OUTPUT_DIR,
    TOKEN_OUTPUT_DIR,
    LOG_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# ------------------------------------------------------------
# 2. Validate Notebook 05 outputs
# ------------------------------------------------------------

required_master_files = {
    "scene_manifest": SCENE_MANIFEST_PATH,
    "scene_profiles": MASTER_PROFILE_PATH,
    "scene_tokens": MASTER_TOKEN_PATH,
}

print("\nNOTEBOOK 05 INPUTS")
print("-" * 72)

for name, path in required_master_files.items():
    print(
        f"{name:<20}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )
    print(f"  {path}")

missing_master_files = [
    name
    for name, path in required_master_files.items()
    if not path.exists()
]

if missing_master_files:
    raise FileNotFoundError(
        "Missing required Notebook 05 outputs: "
        + ", ".join(missing_master_files)
    )

# ------------------------------------------------------------
# 3. Load master transportation tables
# ------------------------------------------------------------

scene_manifest_df = pd.read_csv(
    SCENE_MANIFEST_PATH
)

master_transportation_profiles_df = pd.read_csv(
    MASTER_PROFILE_PATH
)

master_transportation_tokens_df = pd.read_csv(
    MASTER_TOKEN_PATH
)

print("\nLOADED TRANSPORTATION DATA")
print("-" * 72)

print(
    f"Scene manifest records : "
    f"{len(scene_manifest_df):,}"
)

print(
    f"Scene profile records  : "
    f"{len(master_transportation_profiles_df):,}"
)

print(
    f"Scene token records    : "
    f"{len(master_transportation_tokens_df):,}"
)

# ------------------------------------------------------------
# 4. Validate scene alignment
# ------------------------------------------------------------

manifest_scene_ids = set(
    scene_manifest_df["scene_id"]
    .astype(str)
)

profile_scene_ids = set(
    master_transportation_profiles_df["scene_id"]
    .astype(str)
)

token_scene_ids = set(
    master_transportation_tokens_df["scene_id"]
    .astype(str)
)

scene_sets_match = (
    manifest_scene_ids
    == profile_scene_ids
    == token_scene_ids
)

print(
    f"Scene IDs aligned      : "
    f"{scene_sets_match}"
)

if not scene_sets_match:
    raise ValueError(
        "Scene IDs do not match across Notebook 05 master outputs."
    )

# ------------------------------------------------------------
# 5. Inspect candidate raster directories
# ------------------------------------------------------------

candidate_raster_dirs = {
    "S1Hand": S1_DIR,
    "S2Hand": S2_DIR,
    "LabelHand": LABEL_DIR,
    "JRCWaterHand": JRC_WATER_DIR,
}

print("\nCANDIDATE RASTER DIRECTORIES")
print("-" * 72)

for name, path in candidate_raster_dirs.items():

    tif_count = 0

    if path.exists():
        tif_count = len(
            list(path.rglob("*.tif"))
        ) + len(
            list(path.rglob("*.tiff"))
        )

    print(
        f"{name:<16}: "
        f"{'FOUND' if path.exists() else 'MISSING'} "
        f"({tif_count:,} raster files)"
    )

    print(f"  {path}")

# ------------------------------------------------------------
# 6. Inspect road-level outputs
# ------------------------------------------------------------

road_knowledge_files = sorted(
    ROAD_KNOWLEDGE_DIR.glob(
        "*_road_knowledge.gpkg"
    )
)

print("\nROAD KNOWLEDGE FILES")
print("-" * 72)

print(
    f"GeoPackages found      : "
    f"{len(road_knowledge_files):,}"
)

if road_knowledge_files:
    print(
        f"Example file           : "
        f"{road_knowledge_files[0].name}"
    )

# ------------------------------------------------------------
# 7. Final setup validation
# ------------------------------------------------------------

print("\nSETUP SUMMARY")
print("-" * 72)

print(
    f"Transportation scenes  : "
    f"{len(scene_manifest_df):,}"
)

print(
    f"Road knowledge files   : "
    f"{len(road_knowledge_files):,}"
)

print(
    f"Output directory       : "
    f"{ROAD_FLOOD_OUTPUT_DIR}"
)

if len(road_knowledge_files) != len(scene_manifest_df):
    print(
        "\nWARNING: The number of road knowledge files "
        "does not equal the number of manifest scenes."
    )
else:
    print(
        "\nRoad knowledge files align with the scene manifest."
    )

print("=" * 72)

display(
    scene_manifest_df.head()
)

ROAD–FLOOD GROUNDING

NOTEBOOK 05 INPUTS
------------------------------------------------------------------------
scene_manifest      : FOUND
  /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/transportation_scene_manifest.csv
scene_profiles      : FOUND
  /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_profiles.csv
scene_tokens        : FOUND
  /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/master/master_scene_transportation_tokens.csv

LOADED TRANSPORTATION DATA
------------------------------------------------------------------------
Scene manifest records : 26
Scene profile records  : 26
Scene token records    : 26
Scene IDs aligned      : True

CANDIDATE RASTER DIRECTORIES
------------------------------------------------------------------------
S1Hand          : MISSING (0 raster files)
  /home/adjeiowusu1/myproject/ResilientVLM/data/processed

,scene_id,road_records,graph_nodes,graph_edges,critical_edges,single_access_edges,bridge_bottlenecks,major_road_bottlenecks,scene_token_length,processing_seconds
0,Ghana_141910,2,2,1,2,2,0,0,267,0.09
1,India_1018327,54,25,26,12,38,2,2,278,0.08
2,India_1050276,186,81,97,45,71,10,4,287,0.22
3,India_1068117,111,51,60,23,50,2,12,283,0.13
4,India_285297,68,35,40,14,23,3,7,287,0.10


In [2]:
# ============================================================
# CELL 2: DISCOVER SEN1FLOODS11 RASTER LOCATIONS
# ============================================================

from pathlib import Path
import pandas as pd

print("=" * 72)
print("DISCOVERING SEN1FLOODS11 RASTER FILES")
print("=" * 72)

# ------------------------------------------------------------
# 1. Search all raster files under the project data directory
# ------------------------------------------------------------

all_tif_files = sorted(
    list(DATA_DIR.rglob("*.tif"))
    + list(DATA_DIR.rglob("*.tiff"))
)

print(f"\nTotal raster files found: {len(all_tif_files):,}")

if not all_tif_files:
    raise FileNotFoundError(
        f"No .tif or .tiff files were found under:\n{DATA_DIR}"
    )

# ------------------------------------------------------------
# 2. Classify raster files by SEN1Floods11 component
# ------------------------------------------------------------

component_keywords = {
    "S1Hand": [
        "s1hand",
        "s1_hand",
    ],
    "S2Hand": [
        "s2hand",
        "s2_hand",
    ],
    "LabelHand": [
        "labelhand",
        "label_hand",
    ],
    "JRCWaterHand": [
        "jrcwaterhand",
        "jrc_water_hand",
        "jrcwater",
    ],
}

raster_records = []

for raster_path in all_tif_files:

    path_text = str(raster_path).lower()
    filename_text = raster_path.name.lower()

    detected_component = "Unknown"

    for component, keywords in component_keywords.items():

        if any(
            keyword in path_text
            or keyword in filename_text
            for keyword in keywords
        ):
            detected_component = component
            break

    raster_records.append(
        {
            "filename": raster_path.name,
            "component": detected_component,
            "parent_directory": str(raster_path.parent),
            "raster_path": str(raster_path),
        }
    )

raster_inventory_df = pd.DataFrame(
    raster_records
)

# ------------------------------------------------------------
# 3. Display component counts
# ------------------------------------------------------------

print("\nRASTER COMPONENT COUNTS")
print("-" * 72)

component_counts = (
    raster_inventory_df["component"]
    .value_counts(dropna=False)
)

print(component_counts.to_string())

# ------------------------------------------------------------
# 4. Display unique raster directories
# ------------------------------------------------------------

raster_directory_summary_df = (
    raster_inventory_df
    .groupby(
        [
            "component",
            "parent_directory",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="file_count")
    .sort_values(
        [
            "component",
            "file_count",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

print("\nRASTER DIRECTORY SUMMARY")
print("-" * 72)

display(
    raster_directory_summary_df
)

# ------------------------------------------------------------
# 5. Match rasters to the 25 transportation scenes
# ------------------------------------------------------------

scene_ids = (
    scene_manifest_df["scene_id"]
    .astype(str)
    .tolist()
)

scene_raster_records = []

for scene_id in scene_ids:

    scene_matches = raster_inventory_df[
        raster_inventory_df["raster_path"]
        .str.contains(
            scene_id,
            case=False,
            regex=False,
        )
    ].copy()

    component_paths = {}

    for component in component_keywords:

        component_matches = scene_matches[
            scene_matches["component"] == component
        ]

        if not component_matches.empty:
            component_paths[component] = (
                component_matches.iloc[0]["raster_path"]
            )
        else:
            component_paths[component] = None

    scene_raster_records.append(
        {
            "scene_id": scene_id,
            "s1_path": component_paths["S1Hand"],
            "s2_path": component_paths["S2Hand"],
            "label_path": component_paths["LabelHand"],
            "jrc_water_path": component_paths["JRCWaterHand"],
            "s1_exists": component_paths["S1Hand"] is not None,
            "s2_exists": component_paths["S2Hand"] is not None,
            "label_exists": component_paths["LabelHand"] is not None,
            "jrc_water_exists": (
                component_paths["JRCWaterHand"]
                is not None
            ),
        }
    )

scene_raster_inventory_df = pd.DataFrame(
    scene_raster_records
)

scene_raster_inventory_df["complete_raster_set"] = (
    scene_raster_inventory_df[
        [
            "s1_exists",
            "s2_exists",
            "label_exists",
            "jrc_water_exists",
        ]
    ]
    .all(axis=1)
)

# ------------------------------------------------------------
# 6. Validate scene coverage
# ------------------------------------------------------------

print("\nSCENE RASTER COVERAGE")
print("-" * 72)

print(
    f"S1 rasters found       : "
    f"{scene_raster_inventory_df['s1_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"S2 rasters found       : "
    f"{scene_raster_inventory_df['s2_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"Flood labels found     : "
    f"{scene_raster_inventory_df['label_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"JRC water rasters found: "
    f"{scene_raster_inventory_df['jrc_water_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"Complete raster sets   : "
    f"{scene_raster_inventory_df['complete_raster_set'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

# ------------------------------------------------------------
# 7. Show missing scene components
# ------------------------------------------------------------

missing_raster_scenes_df = (
    scene_raster_inventory_df.loc[
        ~scene_raster_inventory_df[
            "complete_raster_set"
        ]
    ]
    .copy()
)

if not missing_raster_scenes_df.empty:

    print("\nSCENES WITH MISSING RASTERS")
    print("-" * 72)

    display(
        missing_raster_scenes_df[
            [
                "scene_id",
                "s1_exists",
                "s2_exists",
                "label_exists",
                "jrc_water_exists",
            ]
        ]
    )

else:
    print(
        "\nAll transportation scenes have complete raster sets."
    )

# ------------------------------------------------------------
# 8. Save discovered raster inventory
# ------------------------------------------------------------

RASTER_INVENTORY_PATH = (
    ROAD_FLOOD_OUTPUT_DIR
    / "sen1floods11_raster_inventory.csv"
)

SCENE_RASTER_INVENTORY_PATH = (
    ROAD_FLOOD_OUTPUT_DIR
    / "scene_raster_inventory.csv"
)

raster_inventory_df.to_csv(
    RASTER_INVENTORY_PATH,
    index=False,
)

scene_raster_inventory_df.to_csv(
    SCENE_RASTER_INVENTORY_PATH,
    index=False,
)

print("\nSAVED INVENTORIES")
print("-" * 72)

print(RASTER_INVENTORY_PATH)
print(SCENE_RASTER_INVENTORY_PATH)

print("\nSCENE RASTER INVENTORY PREVIEW")
display(
    scene_raster_inventory_df.head()
)

print("=" * 72)

DISCOVERING SEN1FLOODS11 RASTER FILES

Total raster files found: 908

RASTER COMPONENT COUNTS
------------------------------------------------------------------------
component
JRCWaterHand    450
LabelHand       450
S1Hand            4
S2Hand            4

RASTER DIRECTORY SUMMARY
------------------------------------------------------------------------


,component,parent_directory,file_count
0,JRCWaterHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,446
1,JRCWaterHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
2,JRCWaterHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
3,JRCWaterHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
4,JRCWaterHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
5,LabelHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,446
6,LabelHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
7,LabelHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
8,LabelHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1
9,LabelHand,/home/adjeiowusu1/myproject/ResilientVLM/data/...,1



SCENE RASTER COVERAGE
------------------------------------------------------------------------
S1 rasters found       : 0/26
S2 rasters found       : 0/26
Flood labels found     : 26/26
JRC water rasters found: 26/26
Complete raster sets   : 0/26

SCENES WITH MISSING RASTERS
------------------------------------------------------------------------


,scene_id,s1_exists,s2_exists,label_exists,jrc_water_exists
0,Ghana_141910,False,False,True,True
1,India_1018327,False,False,True,True
2,India_1050276,False,False,True,True
3,India_1068117,False,False,True,True
4,India_285297,False,False,True,True
5,India_383430,False,False,True,True
6,India_500266,False,False,True,True
7,India_773682,False,False,True,True
8,India_804466,False,False,True,True
9,India_943439,False,False,True,True



SAVED INVENTORIES
------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/sen1floods11_raster_inventory.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/scene_raster_inventory.csv

SCENE RASTER INVENTORY PREVIEW


,scene_id,s1_path,s2_path,label_path,jrc_water_path,s1_exists,s2_exists,label_exists,jrc_water_exists,complete_raster_set
0,Ghana_141910,None,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,False,False,True,True,False
1,India_1018327,None,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,False,False,True,True,False
2,India_1050276,None,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,False,False,True,True,False
3,India_1068117,None,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,False,False,True,True,False
4,India_285297,None,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,False,False,True,True,False


In [3]:
# ============================================================
# CELL 3: BUILD ROAD–FLOOD GROUNDING INVENTORY
# ============================================================

import ast
import json

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

print("=" * 72)
print("BUILDING ROAD–FLOOD GROUNDING INVENTORY")
print("=" * 72)

# ------------------------------------------------------------
# 1. Define actual requirements for road–flood grounding
# ------------------------------------------------------------

# Grounding requires:
#   1. road knowledge
#   2. LabelHand flood raster
#
# JRCWaterHand is strongly preferred because it helps distinguish
# permanent water from event-related flooding.
#
# S1 and S2 imagery are not required for this notebook.

scene_raster_inventory_df[
    "road_knowledge_path"
] = scene_raster_inventory_df["scene_id"].apply(
    lambda scene_id: str(
        ROAD_KNOWLEDGE_DIR
        / f"{scene_id}_road_knowledge.gpkg"
    )
)

scene_raster_inventory_df[
    "road_knowledge_exists"
] = scene_raster_inventory_df[
    "road_knowledge_path"
].apply(
    lambda path: Path(path).exists()
)

scene_raster_inventory_df[
    "grounding_ready"
] = (
    scene_raster_inventory_df["road_knowledge_exists"]
    & scene_raster_inventory_df["label_exists"]
)

scene_raster_inventory_df[
    "grounding_ready_with_jrc"
] = (
    scene_raster_inventory_df["grounding_ready"]
    & scene_raster_inventory_df["jrc_water_exists"]
)

print("\nGROUNDING READINESS")
print("-" * 72)

print(
    f"Road knowledge files     : "
    f"{scene_raster_inventory_df['road_knowledge_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"Flood labels             : "
    f"{scene_raster_inventory_df['label_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"JRC permanent-water masks: "
    f"{scene_raster_inventory_df['jrc_water_exists'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"Grounding-ready scenes   : "
    f"{scene_raster_inventory_df['grounding_ready'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

print(
    f"Ready with JRC masks     : "
    f"{scene_raster_inventory_df['grounding_ready_with_jrc'].sum():,}/"
    f"{len(scene_raster_inventory_df):,}"
)

# ------------------------------------------------------------
# 2. Select a prototype scene for raster inspection
# ------------------------------------------------------------

PROTOTYPE_SCENE_ID = "Spain_8565131"

prototype_row = (
    scene_raster_inventory_df.loc[
        scene_raster_inventory_df["scene_id"]
        == PROTOTYPE_SCENE_ID
    ]
    .iloc[0]
)

prototype_label_path = Path(
    prototype_row["label_path"]
)

prototype_jrc_path = Path(
    prototype_row["jrc_water_path"]
)

prototype_roads_path = Path(
    prototype_row["road_knowledge_path"]
)

print("\nPROTOTYPE INPUTS")
print("-" * 72)

print(f"Scene       : {PROTOTYPE_SCENE_ID}")
print(f"Flood label : {prototype_label_path}")
print(f"JRC mask    : {prototype_jrc_path}")
print(f"Roads       : {prototype_roads_path}")

# ------------------------------------------------------------
# 3. Inspect flood-label raster
# ------------------------------------------------------------

with rasterio.open(
    prototype_label_path
) as label_src:

    label_array = label_src.read(1)

    label_metadata = {
        "width": label_src.width,
        "height": label_src.height,
        "count": label_src.count,
        "dtype": str(label_src.dtypes[0]),
        "crs": str(label_src.crs),
        "transform": str(label_src.transform),
        "nodata": label_src.nodata,
        "bounds": tuple(label_src.bounds),
    }

    label_unique_values, label_unique_counts = (
        np.unique(
            label_array,
            return_counts=True,
        )
    )

label_value_counts = {
    str(value): int(count)
    for value, count in zip(
        label_unique_values,
        label_unique_counts,
    )
}

print("\nFLOOD-LABEL RASTER")
print("-" * 72)

for key, value in label_metadata.items():
    print(f"{key:<12}: {value}")

print(
    f"Unique values: "
    f"{label_value_counts}"
)

# ------------------------------------------------------------
# 4. Inspect JRC permanent-water raster
# ------------------------------------------------------------

with rasterio.open(
    prototype_jrc_path
) as jrc_src:

    jrc_array = jrc_src.read(1)

    jrc_metadata = {
        "width": jrc_src.width,
        "height": jrc_src.height,
        "count": jrc_src.count,
        "dtype": str(jrc_src.dtypes[0]),
        "crs": str(jrc_src.crs),
        "transform": str(jrc_src.transform),
        "nodata": jrc_src.nodata,
        "bounds": tuple(jrc_src.bounds),
    }

    jrc_unique_values, jrc_unique_counts = (
        np.unique(
            jrc_array,
            return_counts=True,
        )
    )

jrc_value_counts = {
    str(value): int(count)
    for value, count in zip(
        jrc_unique_values,
        jrc_unique_counts,
    )
}

print("\nJRC PERMANENT-WATER RASTER")
print("-" * 72)

for key, value in jrc_metadata.items():
    print(f"{key:<12}: {value}")

print(
    f"Unique values: "
    f"{jrc_value_counts}"
)

# ------------------------------------------------------------
# 5. Inspect prototype road layer
# ------------------------------------------------------------

prototype_roads_gdf = gpd.read_file(
    prototype_roads_path
)

print("\nPROTOTYPE ROAD KNOWLEDGE")
print("-" * 72)

print(
    f"Road records : "
    f"{len(prototype_roads_gdf):,}"
)

print(
    f"Road CRS     : "
    f"{prototype_roads_gdf.crs}"
)

print(
    f"Road bounds  : "
    f"{tuple(prototype_roads_gdf.total_bounds)}"
)

print(
    f"Geometry types: "
    f"{prototype_roads_gdf.geometry.geom_type.value_counts().to_dict()}"
)

# ------------------------------------------------------------
# 6. Validate raster alignment
# ------------------------------------------------------------

same_raster_shape = (
    label_metadata["width"]
    == jrc_metadata["width"]
    and label_metadata["height"]
    == jrc_metadata["height"]
)

same_raster_crs = (
    label_metadata["crs"]
    == jrc_metadata["crs"]
)

same_raster_bounds = np.allclose(
    np.array(label_metadata["bounds"]),
    np.array(jrc_metadata["bounds"]),
)

print("\nRASTER ALIGNMENT")
print("-" * 72)

print(
    f"Same dimensions : "
    f"{same_raster_shape}"
)

print(
    f"Same CRS        : "
    f"{same_raster_crs}"
)

print(
    f"Same bounds     : "
    f"{same_raster_bounds}"
)

if not same_raster_shape:
    print(
        "\nWARNING: Label and JRC rasters have "
        "different dimensions."
    )

if not same_raster_crs:
    print(
        "\nWARNING: Label and JRC rasters have "
        "different coordinate reference systems."
    )

if not same_raster_bounds:
    print(
        "\nWARNING: Label and JRC rasters have "
        "different spatial extents."
    )

# ------------------------------------------------------------
# 7. Check road/raster CRS compatibility
# ------------------------------------------------------------

with rasterio.open(
    prototype_label_path
) as label_src:
    label_crs = label_src.crs

road_crs_matches_label = (
    prototype_roads_gdf.crs is not None
    and label_crs is not None
    and prototype_roads_gdf.crs.to_string()
    == label_crs.to_string()
)

print("\nROAD–RASTER CRS")
print("-" * 72)

print(
    f"Road CRS matches raster CRS: "
    f"{road_crs_matches_label}"
)

if not road_crs_matches_label:
    print(
        "Road geometries will be reprojected to the "
        "label-raster CRS during grounding."
    )

# ------------------------------------------------------------
# 8. Create compact grounding inventory
# ------------------------------------------------------------

grounding_inventory_df = (
    scene_raster_inventory_df.loc[
        scene_raster_inventory_df[
            "grounding_ready"
        ]
    ]
    [
        [
            "scene_id",
            "road_knowledge_path",
            "label_path",
            "jrc_water_path",
            "s1_path",
            "s2_path",
            "road_knowledge_exists",
            "label_exists",
            "jrc_water_exists",
            "s1_exists",
            "s2_exists",
            "grounding_ready",
            "grounding_ready_with_jrc",
        ]
    ]
    .copy()
    .sort_values("scene_id")
    .reset_index(drop=True)
)

GROUNDING_INVENTORY_PATH = (
    ROAD_FLOOD_OUTPUT_DIR
    / "road_flood_grounding_inventory.csv"
)

grounding_inventory_df.to_csv(
    GROUNDING_INVENTORY_PATH,
    index=False,
)

print("\nGROUNDING INVENTORY SAVED")
print("-" * 72)

print(GROUNDING_INVENTORY_PATH)

print("\nGROUNDING INVENTORY PREVIEW")
display(
    grounding_inventory_df.head()
)

print("=" * 72)

BUILDING ROAD–FLOOD GROUNDING INVENTORY

GROUNDING READINESS
------------------------------------------------------------------------
Road knowledge files     : 26/26
Flood labels             : 26/26
JRC permanent-water masks: 26/26
Grounding-ready scenes   : 26/26
Ready with JRC masks     : 26/26

PROTOTYPE INPUTS
------------------------------------------------------------------------
Scene       : Spain_8565131
Flood label : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/LabelHand/Spain_8565131_LabelHand.tif
JRC mask    : /home/adjeiowusu1/myproject/ResilientVLM/data/raw/sen1floods11/hand_labeled/JRCWaterHand/Spain_8565131_JRCWaterHand.tif
Roads       : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/transportation_knowledge/road_knowledge/Spain_8565131_road_knowledge.gpkg

FLOOD-LABEL RASTER
------------------------------------------------------------------------
width       : 512
height      : 512
count       : 1
dtype       : int16
crs        


PROTOTYPE ROAD KNOWLEDGE
------------------------------------------------------------------------
Road records : 902
Road CRS     : EPSG:4326
Road bounds  : (np.float64(-0.8419834), np.float64(38.1696867), np.float64(-0.7715215), np.float64(38.2278547))
Geometry types: {'LineString': 902}

RASTER ALIGNMENT
------------------------------------------------------------------------
Same dimensions : True
Same CRS        : True
Same bounds     : True

ROAD–RASTER CRS
------------------------------------------------------------------------
Road CRS matches raster CRS: True

GROUNDING INVENTORY SAVED
------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/road_flood_grounding_inventory.csv

GROUNDING INVENTORY PREVIEW


,scene_id,road_knowledge_path,label_path,jrc_water_path,s1_path,s2_path,road_knowledge_exists,label_exists,jrc_water_exists,s1_exists,s2_exists,grounding_ready,grounding_ready_with_jrc
0,Ghana_141910,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,None,None,True,True,True,False,False,True,True
1,India_1018327,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,None,None,True,True,True,False,False,True,True
2,India_1050276,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,None,None,True,True,True,False,False,True,True
3,India_1068117,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,None,None,True,True,True,False,False,True,True
4,India_285297,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...,None,None,True,True,True,False,False,True,True


In [4]:
# ============================================================
# CELL 4: PROTOTYPE ROAD–FLOOD SPATIAL GROUNDING
# ============================================================

from shapely.geometry import box
from rasterio.features import rasterize
import time

print("=" * 72)
print("PROTOTYPE ROAD–FLOOD SPATIAL GROUNDING")
print("=" * 72)

# ------------------------------------------------------------
# 1. Define SEN1Floods11 class values
# ------------------------------------------------------------

LABEL_INVALID_VALUE = -1
LABEL_NOT_WATER_VALUE = 0
LABEL_WATER_VALUE = 1

JRC_NOT_PERMANENT_WATER_VALUE = 0
JRC_PERMANENT_WATER_VALUE = 1

# A 10-meter buffer approximates one SEN1Floods11 pixel and
# reduces missed intersections caused by raster/road alignment.
ROAD_BUFFER_METERS = 10.0

# Thresholds used only for initial semantic categorization.
FLOOD_TOUCH_THRESHOLD = 0.01
MODERATE_FLOOD_THRESHOLD = 0.10
HIGH_FLOOD_THRESHOLD = 0.30

# ------------------------------------------------------------
# 2. Load prototype raster arrays
# ------------------------------------------------------------

with rasterio.open(
    prototype_label_path
) as label_src:

    label_array = label_src.read(1)

    raster_transform = label_src.transform
    raster_crs = label_src.crs
    raster_width = label_src.width
    raster_height = label_src.height
    raster_bounds = label_src.bounds

with rasterio.open(
    prototype_jrc_path
) as jrc_src:

    jrc_array = jrc_src.read(1)

# ------------------------------------------------------------
# 3. Construct semantic water masks
# ------------------------------------------------------------

valid_mask = (
    label_array != LABEL_INVALID_VALUE
)

surface_water_mask = (
    (label_array == LABEL_WATER_VALUE)
    & valid_mask
)

permanent_water_mask = (
    (jrc_array == JRC_PERMANENT_WATER_VALUE)
    & valid_mask
)

event_flood_proxy_mask = (
    surface_water_mask
    & ~permanent_water_mask
)

non_water_mask = (
    (label_array == LABEL_NOT_WATER_VALUE)
    & valid_mask
)

print("\nPROTOTYPE PIXEL CLASSES")
print("-" * 72)

print(
    f"Valid pixels          : "
    f"{valid_mask.sum():,}"
)

print(
    f"Invalid pixels        : "
    f"{(~valid_mask).sum():,}"
)

print(
    f"Surface-water pixels  : "
    f"{surface_water_mask.sum():,}"
)

print(
    f"Permanent-water pixels: "
    f"{permanent_water_mask.sum():,}"
)

print(
    f"Event-flood proxy     : "
    f"{event_flood_proxy_mask.sum():,}"
)

print(
    f"Non-water pixels      : "
    f"{non_water_mask.sum():,}"
)

# ------------------------------------------------------------
# 4. Validate the relationship between masks
# ------------------------------------------------------------

jrc_outside_label_water = (
    permanent_water_mask
    & ~surface_water_mask
)

print("\nMASK CONSISTENCY")
print("-" * 72)

print(
    f"JRC pixels outside LabelHand water: "
    f"{jrc_outside_label_water.sum():,}"
)

print(
    f"Surface-water share of valid pixels: "
    f"{surface_water_mask.sum() / valid_mask.sum():.4f}"
)

print(
    f"Event-flood proxy share of valid pixels: "
    f"{event_flood_proxy_mask.sum() / valid_mask.sum():.4f}"
)

# ------------------------------------------------------------
# 5. Prepare road geometries
# ------------------------------------------------------------

roads_grounding_gdf = prototype_roads_gdf.copy()

roads_grounding_gdf = roads_grounding_gdf.loc[
    roads_grounding_gdf.geometry.notna()
    & ~roads_grounding_gdf.geometry.is_empty
].copy()

# Preserve the original line geometry.
roads_grounding_gdf[
    "road_line_geometry"
] = roads_grounding_gdf.geometry.copy()

# Determine an appropriate projected CRS for meter-based buffering.
projected_crs = (
    roads_grounding_gdf.estimate_utm_crs()
)

if projected_crs is None:
    raise ValueError(
        "A projected CRS could not be estimated "
        "for the prototype road network."
    )

print("\nROAD BUFFER PREPARATION")
print("-" * 72)

print(
    f"Original CRS       : "
    f"{roads_grounding_gdf.crs}"
)

print(
    f"Projected CRS      : "
    f"{projected_crs}"
)

print(
    f"Road buffer        : "
    f"{ROAD_BUFFER_METERS:.1f} meters"
)

# Project, buffer, and return to raster CRS.
roads_projected_gdf = (
    roads_grounding_gdf
    .to_crs(projected_crs)
)

buffered_projected_gdf = (
    roads_projected_gdf.copy()
)

buffered_projected_gdf.geometry = (
    buffered_projected_gdf.geometry.buffer(
        ROAD_BUFFER_METERS
    )
)

buffered_roads_gdf = (
    buffered_projected_gdf
    .to_crs(raster_crs)
)

# ------------------------------------------------------------
# 6. Clip buffered roads to raster extent
# ------------------------------------------------------------

raster_extent_geometry = box(
    raster_bounds.left,
    raster_bounds.bottom,
    raster_bounds.right,
    raster_bounds.top,
)

buffered_roads_gdf[
    "intersects_raster"
] = buffered_roads_gdf.geometry.intersects(
    raster_extent_geometry
)

buffered_roads_gdf[
    "buffered_geometry"
] = buffered_roads_gdf.geometry.copy()

buffered_roads_gdf.geometry = (
    buffered_roads_gdf.geometry.intersection(
        raster_extent_geometry
    )
)

print("\nROAD–RASTER COVERAGE")
print("-" * 72)

print(
    f"Total road records        : "
    f"{len(buffered_roads_gdf):,}"
)

print(
    f"Roads intersecting raster : "
    f"{buffered_roads_gdf['intersects_raster'].sum():,}"
)

print(
    f"Roads outside raster      : "
    f"{(~buffered_roads_gdf['intersects_raster']).sum():,}"
)

# ------------------------------------------------------------
# 7. Ground individual roads to raster pixels
# ------------------------------------------------------------

def classify_road_flood_exposure(
    event_flood_fraction,
    valid_pixel_count,
):
    """
    Assign an initial semantic flood-exposure class based on
    the fraction of valid road-corridor pixels classified as
    event-flood proxy pixels.
    """

    if valid_pixel_count == 0:
        return "No Valid Raster Coverage"

    if event_flood_fraction < FLOOD_TOUCH_THRESHOLD:
        return "No Detected Flood Exposure"

    if event_flood_fraction < MODERATE_FLOOD_THRESHOLD:
        return "Low Flood Exposure"

    if event_flood_fraction < HIGH_FLOOD_THRESHOLD:
        return "Moderate Flood Exposure"

    return "High Flood Exposure"


road_flood_records = []

grounding_start_time = time.time()

total_roads = len(buffered_roads_gdf)

for road_number, (road_index, road_row) in enumerate(
    buffered_roads_gdf.iterrows(),
    start=1,
):

    road_geometry = road_row.geometry

    record = {
        "road_index": road_index,
        "intersects_raster": bool(
            road_row["intersects_raster"]
        ),
        "corridor_pixel_count": 0,
        "valid_pixel_count": 0,
        "invalid_pixel_count": 0,
        "surface_water_pixel_count": 0,
        "permanent_water_pixel_count": 0,
        "event_flood_pixel_count": 0,
        "non_water_pixel_count": 0,
        "valid_coverage_fraction": 0.0,
        "surface_water_fraction": 0.0,
        "permanent_water_fraction": 0.0,
        "event_flood_fraction": 0.0,
        "non_water_fraction": 0.0,
        "touches_surface_water": False,
        "touches_permanent_water": False,
        "touches_event_flood": False,
    }

    if (
        road_geometry is None
        or road_geometry.is_empty
        or not road_row["intersects_raster"]
    ):
        record["road_flood_exposure_class"] = (
            "Outside Raster Coverage"
        )

        road_flood_records.append(record)
        continue

    # Rasterize the buffered road corridor.
    road_corridor_mask = rasterize(
        [
            (
                road_geometry,
                1,
            )
        ],
        out_shape=(
            raster_height,
            raster_width,
        ),
        transform=raster_transform,
        fill=0,
        all_touched=True,
        dtype="uint8",
    ).astype(bool)

    corridor_pixel_count = int(
        road_corridor_mask.sum()
    )

    valid_road_pixels = (
        road_corridor_mask
        & valid_mask
    )

    invalid_road_pixels = (
        road_corridor_mask
        & ~valid_mask
    )

    surface_water_road_pixels = (
        road_corridor_mask
        & surface_water_mask
    )

    permanent_water_road_pixels = (
        road_corridor_mask
        & permanent_water_mask
    )

    event_flood_road_pixels = (
        road_corridor_mask
        & event_flood_proxy_mask
    )

    non_water_road_pixels = (
        road_corridor_mask
        & non_water_mask
    )

    valid_pixel_count = int(
        valid_road_pixels.sum()
    )

    invalid_pixel_count = int(
        invalid_road_pixels.sum()
    )

    surface_water_pixel_count = int(
        surface_water_road_pixels.sum()
    )

    permanent_water_pixel_count = int(
        permanent_water_road_pixels.sum()
    )

    event_flood_pixel_count = int(
        event_flood_road_pixels.sum()
    )

    non_water_pixel_count = int(
        non_water_road_pixels.sum()
    )

    if corridor_pixel_count > 0:
        valid_coverage_fraction = (
            valid_pixel_count
            / corridor_pixel_count
        )
    else:
        valid_coverage_fraction = 0.0

    if valid_pixel_count > 0:

        surface_water_fraction = (
            surface_water_pixel_count
            / valid_pixel_count
        )

        permanent_water_fraction = (
            permanent_water_pixel_count
            / valid_pixel_count
        )

        event_flood_fraction = (
            event_flood_pixel_count
            / valid_pixel_count
        )

        non_water_fraction = (
            non_water_pixel_count
            / valid_pixel_count
        )

    else:
        surface_water_fraction = 0.0
        permanent_water_fraction = 0.0
        event_flood_fraction = 0.0
        non_water_fraction = 0.0

    record.update(
        {
            "corridor_pixel_count": corridor_pixel_count,
            "valid_pixel_count": valid_pixel_count,
            "invalid_pixel_count": invalid_pixel_count,
            "surface_water_pixel_count": (
                surface_water_pixel_count
            ),
            "permanent_water_pixel_count": (
                permanent_water_pixel_count
            ),
            "event_flood_pixel_count": (
                event_flood_pixel_count
            ),
            "non_water_pixel_count": (
                non_water_pixel_count
            ),
            "valid_coverage_fraction": round(
                valid_coverage_fraction,
                4,
            ),
            "surface_water_fraction": round(
                surface_water_fraction,
                4,
            ),
            "permanent_water_fraction": round(
                permanent_water_fraction,
                4,
            ),
            "event_flood_fraction": round(
                event_flood_fraction,
                4,
            ),
            "non_water_fraction": round(
                non_water_fraction,
                4,
            ),
            "touches_surface_water": (
                surface_water_pixel_count > 0
            ),
            "touches_permanent_water": (
                permanent_water_pixel_count > 0
            ),
            "touches_event_flood": (
                event_flood_pixel_count > 0
            ),
            "road_flood_exposure_class": (
                classify_road_flood_exposure(
                    event_flood_fraction,
                    valid_pixel_count,
                )
            ),
        }
    )

    road_flood_records.append(record)

    if (
        road_number % 100 == 0
        or road_number == total_roads
    ):
        print(
            f"Processed "
            f"{road_number:,}/{total_roads:,} roads"
        )

grounding_runtime_seconds = (
    time.time() - grounding_start_time
)

# ------------------------------------------------------------
# 8. Join flood metrics back to road knowledge
# ------------------------------------------------------------

road_flood_metrics_df = pd.DataFrame(
    road_flood_records
).set_index(
    "road_index"
)

prototype_road_flood_gdf = (
    roads_grounding_gdf
    .join(
        road_flood_metrics_df,
        how="left",
    )
)

# ------------------------------------------------------------
# 9. Add grounded semantic tokens
# ------------------------------------------------------------

def build_road_flood_token(row):

    exposure_class = str(
        row.get(
            "road_flood_exposure_class",
            "Unknown",
        )
    ).replace(
        " ",
        ""
    )

    surface_water_value = (
        "Yes"
        if bool(
            row.get(
                "touches_surface_water",
                False,
            )
        )
        else "No"
    )

    permanent_water_value = (
        "Yes"
        if bool(
            row.get(
                "touches_permanent_water",
                False,
            )
        )
        else "No"
    )

    flood_value = (
        "Yes"
        if bool(
            row.get(
                "touches_event_flood",
                False,
            )
        )
        else "No"
    )

    return " ".join(
        [
            (
                f"<RoadFloodExposure:"
                f"{exposure_class}>"
            ),
            (
                f"<TouchesSurfaceWater:"
                f"{surface_water_value}>"
            ),
            (
                f"<TouchesPermanentWater:"
                f"{permanent_water_value}>"
            ),
            (
                f"<TouchesEventFlood:"
                f"{flood_value}>"
            ),
        ]
    )


prototype_road_flood_gdf[
    "road_flood_token"
] = prototype_road_flood_gdf.apply(
    build_road_flood_token,
    axis=1,
)

# ------------------------------------------------------------
# 10. Prototype summary
# ------------------------------------------------------------

roads_with_valid_coverage = int(
    (
        prototype_road_flood_gdf[
            "valid_pixel_count"
        ] > 0
    ).sum()
)

roads_touching_surface_water = int(
    prototype_road_flood_gdf[
        "touches_surface_water"
    ].fillna(False).sum()
)

roads_touching_permanent_water = int(
    prototype_road_flood_gdf[
        "touches_permanent_water"
    ].fillna(False).sum()
)

roads_touching_event_flood = int(
    prototype_road_flood_gdf[
        "touches_event_flood"
    ].fillna(False).sum()
)

print("\nPROTOTYPE ROAD–FLOOD SUMMARY")
print("-" * 72)

print(
    f"Road records                 : "
    f"{len(prototype_road_flood_gdf):,}"
)

print(
    f"Roads with valid coverage    : "
    f"{roads_with_valid_coverage:,}"
)

print(
    f"Roads touching surface water : "
    f"{roads_touching_surface_water:,}"
)

print(
    f"Roads touching permanent water: "
    f"{roads_touching_permanent_water:,}"
)

print(
    f"Roads touching event flood   : "
    f"{roads_touching_event_flood:,}"
)

print(
    f"Processing time              : "
    f"{grounding_runtime_seconds:.2f} seconds"
)

print("\nEXPOSURE CLASS DISTRIBUTION")
print("-" * 72)

print(
    prototype_road_flood_gdf[
        "road_flood_exposure_class"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)

# ------------------------------------------------------------
# 11. Display most exposed roads
# ------------------------------------------------------------

preview_columns = [
    column
    for column in [
        "u",
        "v",
        "name",
        "highway",
        "hierarchy_group",
        "network_role",
        "is_physical_bridge",
        "is_critical_transport_edge",
        "is_critical_low_redundancy",
        "event_flood_pixel_count",
        "event_flood_fraction",
        "permanent_water_fraction",
        "road_flood_exposure_class",
        "road_flood_token",
    ]
    if column in prototype_road_flood_gdf.columns
]

print("\nMOST EXPOSED PROTOTYPE ROADS")
print("-" * 72)

display(
    prototype_road_flood_gdf[
        preview_columns
    ]
    .sort_values(
        "event_flood_fraction",
        ascending=False,
    )
    .head(15)
)

print("=" * 72)

PROTOTYPE ROAD–FLOOD SPATIAL GROUNDING



PROTOTYPE PIXEL CLASSES
------------------------------------------------------------------------
Valid pixels          : 262,084
Invalid pixels        : 60
Surface-water pixels  : 52,356
Permanent-water pixels: 996
Event-flood proxy     : 51,431
Non-water pixels      : 209,728

MASK CONSISTENCY
------------------------------------------------------------------------
JRC pixels outside LabelHand water: 71
Surface-water share of valid pixels: 0.1998
Event-flood proxy share of valid pixels: 0.1962



ROAD BUFFER PREPARATION
------------------------------------------------------------------------
Original CRS       : EPSG:4326
Projected CRS      : EPSG:32630
Road buffer        : 10.0 meters



ROAD–RASTER COVERAGE
------------------------------------------------------------------------
Total road records        : 902
Roads intersecting raster : 775
Roads outside raster      : 127


Processed 100/902 roads


...

Processed 500/902 roads


Processed 600/902 roads


Processed 700/902 roads



PROTOTYPE ROAD–FLOOD SUMMARY
------------------------------------------------------------------------
Road records                 : 902
Roads with valid coverage    : 775
Roads touching surface water : 104
Roads touching permanent water: 0
Roads touching event flood   : 104
Processing time              : 1.35 seconds

EXPOSURE CLASS DISTRIBUTION
------------------------------------------------------------------------
road_flood_exposure_class
No Detected Flood Exposure    675
Outside Raster Coverage       127
Low Flood Exposure             48
High Flood Exposure            30
Moderate Flood Exposure        22

MOST EXPOSED PROTOTYPE ROADS
------------------------------------------------------------------------


,u,v,name,highway,hierarchy_group,event_flood_pixel_count,event_flood_fraction,permanent_water_fraction,road_flood_exposure_class,road_flood_token
539,2665217835,2665217834,None,unclassified,local,95,0.7090,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
538,2665217834,2665217835,None,unclassified,local,95,0.7090,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
686,6138927629,6555102393,None,unclassified,local,257,0.6946,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
702,6555102393,6138927629,None,unclassified,local,257,0.6946,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
723,6576753658,6576753662,None,residential,local,50,0.6757,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
725,6576753662,6576753658,None,residential,local,50,0.6757,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
720,6576753657,6576753671,None,unclassified,local,156,0.6473,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
726,6576753671,6576753657,None,unclassified,local,156,0.6473,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
380,1797999888,406556546,Carretera de Crevillent a Catral,tertiary,local_collector,98,0.6049,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...
15,406556546,1797999888,Carretera de Crevillent a Catral,tertiary,local_collector,98,0.6049,0.0,High Flood Exposure,<RoadFloodExposure:HighFloodExposure> <Touches...


In [5]:
# ============================================================
# CELL 5: PROTOTYPE PHYSICAL-EDGE AND SCENE-LEVEL SUMMARY
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("BUILDING PROTOTYPE ROAD–FLOOD SCENE SUMMARY")
print("=" * 72)

prototype_summary_gdf = prototype_road_flood_gdf.copy()

# ------------------------------------------------------------
# 1. Create an undirected physical-edge identifier
# ------------------------------------------------------------

def build_physical_edge_id(row):
    """
    Create the same identifier for u->v and v->u records.
    """

    u_value = str(row.get("u", ""))
    v_value = str(row.get("v", ""))

    endpoint_1, endpoint_2 = sorted(
        [u_value, v_value]
    )

    key_value = str(
        row.get("key", 0)
    )

    return (
        f"{endpoint_1}__"
        f"{endpoint_2}__"
        f"{key_value}"
    )


prototype_summary_gdf[
    "physical_edge_id"
] = prototype_summary_gdf.apply(
    build_physical_edge_id,
    axis=1,
)

print("\nDIRECTED AND PHYSICAL EDGE COUNTS")
print("-" * 72)

print(
    f"Directed road records : "
    f"{len(prototype_summary_gdf):,}"
)

print(
    f"Unique physical edges : "
    f"{prototype_summary_gdf['physical_edge_id'].nunique():,}"
)

# ------------------------------------------------------------
# 2. Collapse directed duplicates to physical road segments
# ------------------------------------------------------------

aggregation_rules = {
    "intersects_raster": "max",
    "corridor_pixel_count": "max",
    "valid_pixel_count": "max",
    "invalid_pixel_count": "max",
    "surface_water_pixel_count": "max",
    "permanent_water_pixel_count": "max",
    "event_flood_pixel_count": "max",
    "non_water_pixel_count": "max",
    "valid_coverage_fraction": "max",
    "surface_water_fraction": "max",
    "permanent_water_fraction": "max",
    "event_flood_fraction": "max",
    "non_water_fraction": "max",
    "touches_surface_water": "max",
    "touches_permanent_water": "max",
    "touches_event_flood": "max",
    "road_flood_exposure_class": "first",
    "road_flood_token": "first",
}

optional_first_columns = [
    "name",
    "highway",
    "hierarchy_group",
    "network_role",
    "vulnerability_label",
]

optional_max_columns = [
    "is_physical_bridge",
    "is_graph_bridge",
    "is_major_road",
    "is_critical_transport_edge",
    "is_critical_low_redundancy",
    "is_bridge_bottleneck",
    "is_major_road_bottleneck",
    "topological_vulnerability_score",
    "edge_criticality_score",
]

for column in optional_first_columns:
    if column in prototype_summary_gdf.columns:
        aggregation_rules[column] = "first"

for column in optional_max_columns:
    if column in prototype_summary_gdf.columns:
        aggregation_rules[column] = "max"

prototype_physical_edges_df = (
    prototype_summary_gdf
    .groupby(
        "physical_edge_id",
        as_index=False,
    )
    .agg(aggregation_rules)
)

# ------------------------------------------------------------
# 3. Scene-level road coverage metrics
# ------------------------------------------------------------

total_physical_edges = len(
    prototype_physical_edges_df
)

edges_inside_raster = int(
    prototype_physical_edges_df[
        "intersects_raster"
    ]
    .fillna(False)
    .sum()
)

edges_outside_raster = (
    total_physical_edges
    - edges_inside_raster
)

edges_with_valid_coverage = int(
    (
        prototype_physical_edges_df[
            "valid_pixel_count"
        ] > 0
    ).sum()
)

surface_water_edges = int(
    prototype_physical_edges_df[
        "touches_surface_water"
    ]
    .fillna(False)
    .sum()
)

permanent_water_edges = int(
    prototype_physical_edges_df[
        "touches_permanent_water"
    ]
    .fillna(False)
    .sum()
)

event_flood_edges = int(
    prototype_physical_edges_df[
        "touches_event_flood"
    ]
    .fillna(False)
    .sum()
)

low_flood_edges = int(
    (
        prototype_physical_edges_df[
            "road_flood_exposure_class"
        ] == "Low Flood Exposure"
    ).sum()
)

moderate_flood_edges = int(
    (
        prototype_physical_edges_df[
            "road_flood_exposure_class"
        ] == "Moderate Flood Exposure"
    ).sum()
)

high_flood_edges = int(
    (
        prototype_physical_edges_df[
            "road_flood_exposure_class"
        ] == "High Flood Exposure"
    ).sum()
)

# ------------------------------------------------------------
# 4. Critical-infrastructure flood metrics
# ------------------------------------------------------------

def boolean_count(df, column):
    if column not in df.columns:
        return 0

    return int(
        df[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )


def flooded_boolean_count(
    df,
    infrastructure_column,
):
    if infrastructure_column not in df.columns:
        return 0

    return int(
        (
            df[infrastructure_column]
            .fillna(False)
            .astype(bool)
            &
            df["touches_event_flood"]
            .fillna(False)
            .astype(bool)
        ).sum()
    )


critical_edge_count = boolean_count(
    prototype_physical_edges_df,
    "is_critical_transport_edge",
)

flooded_critical_edge_count = flooded_boolean_count(
    prototype_physical_edges_df,
    "is_critical_transport_edge",
)

critical_low_redundancy_count = boolean_count(
    prototype_physical_edges_df,
    "is_critical_low_redundancy",
)

flooded_critical_low_redundancy_count = (
    flooded_boolean_count(
        prototype_physical_edges_df,
        "is_critical_low_redundancy",
    )
)

bridge_bottleneck_count = boolean_count(
    prototype_physical_edges_df,
    "is_bridge_bottleneck",
)

flooded_bridge_bottleneck_count = (
    flooded_boolean_count(
        prototype_physical_edges_df,
        "is_bridge_bottleneck",
    )
)

major_road_bottleneck_count = boolean_count(
    prototype_physical_edges_df,
    "is_major_road_bottleneck",
)

flooded_major_road_bottleneck_count = (
    flooded_boolean_count(
        prototype_physical_edges_df,
        "is_major_road_bottleneck",
    )
)

physical_bridge_count = boolean_count(
    prototype_physical_edges_df,
    "is_physical_bridge",
)

flooded_physical_bridge_count = (
    flooded_boolean_count(
        prototype_physical_edges_df,
        "is_physical_bridge",
    )
)

# ------------------------------------------------------------
# 5. Helper for safe shares
# ------------------------------------------------------------

def safe_share(numerator, denominator):
    if denominator == 0:
        return 0.0

    return round(
        numerator / denominator,
        4,
    )

# ------------------------------------------------------------
# 6. Create scene-level summary
# ------------------------------------------------------------

prototype_scene_flood_profile = {
    "scene_id": PROTOTYPE_SCENE_ID,

    # Record structure
    "directed_road_record_count": int(
        len(prototype_summary_gdf)
    ),
    "physical_edge_count": int(
        total_physical_edges
    ),

    # Raster coverage
    "edges_inside_raster_count": int(
        edges_inside_raster
    ),
    "edges_outside_raster_count": int(
        edges_outside_raster
    ),
    "edges_with_valid_coverage_count": int(
        edges_with_valid_coverage
    ),
    "raster_coverage_share": safe_share(
        edges_inside_raster,
        total_physical_edges,
    ),

    # General flood exposure
    "surface_water_edge_count": int(
        surface_water_edges
    ),
    "permanent_water_edge_count": int(
        permanent_water_edges
    ),
    "event_flood_edge_count": int(
        event_flood_edges
    ),
    "event_flood_edge_share": safe_share(
        event_flood_edges,
        edges_with_valid_coverage,
    ),

    # Exposure severity
    "low_flood_exposure_edge_count": int(
        low_flood_edges
    ),
    "moderate_flood_exposure_edge_count": int(
        moderate_flood_edges
    ),
    "high_flood_exposure_edge_count": int(
        high_flood_edges
    ),

    # Critical transportation infrastructure
    "critical_edge_count": int(
        critical_edge_count
    ),
    "flooded_critical_edge_count": int(
        flooded_critical_edge_count
    ),
    "flooded_critical_edge_share": safe_share(
        flooded_critical_edge_count,
        critical_edge_count,
    ),

    "critical_low_redundancy_count": int(
        critical_low_redundancy_count
    ),
    "flooded_critical_low_redundancy_count": int(
        flooded_critical_low_redundancy_count
    ),
    "flooded_critical_low_redundancy_share": safe_share(
        flooded_critical_low_redundancy_count,
        critical_low_redundancy_count,
    ),

    "physical_bridge_count": int(
        physical_bridge_count
    ),
    "flooded_physical_bridge_count": int(
        flooded_physical_bridge_count
    ),

    "bridge_bottleneck_count": int(
        bridge_bottleneck_count
    ),
    "flooded_bridge_bottleneck_count": int(
        flooded_bridge_bottleneck_count
    ),

    "major_road_bottleneck_count": int(
        major_road_bottleneck_count
    ),
    "flooded_major_road_bottleneck_count": int(
        flooded_major_road_bottleneck_count
    ),

    # Continuous exposure metrics
    "mean_event_flood_fraction": round(
        float(
            prototype_physical_edges_df.loc[
                prototype_physical_edges_df[
                    "valid_pixel_count"
                ] > 0,
                "event_flood_fraction",
            ].mean()
        ),
        4,
    ),

    "maximum_event_flood_fraction": round(
        float(
            prototype_physical_edges_df[
                "event_flood_fraction"
            ].max()
        ),
        4,
    ),
}

prototype_scene_flood_profile_df = pd.DataFrame(
    [prototype_scene_flood_profile]
)

# ------------------------------------------------------------
# 7. Scene-level semantic classifications
# ------------------------------------------------------------

def classify_scene_flood_burden(
    flood_edge_share,
):
    if flood_edge_share < 0.01:
        return "Minimal"

    if flood_edge_share < 0.10:
        return "Low"

    if flood_edge_share < 0.25:
        return "Moderate"

    return "High"


def classify_critical_disruption(
    flooded_critical_share,
    flooded_bottleneck_count,
):
    if (
        flooded_critical_share == 0
        and flooded_bottleneck_count == 0
    ):
        return "None"

    if (
        flooded_critical_share < 0.10
        and flooded_bottleneck_count == 0
    ):
        return "Low"

    if (
        flooded_critical_share < 0.30
        and flooded_bottleneck_count <= 2
    ):
        return "Moderate"

    return "High"


scene_flood_burden = classify_scene_flood_burden(
    prototype_scene_flood_profile[
        "event_flood_edge_share"
    ]
)

critical_disruption_class = (
    classify_critical_disruption(
        prototype_scene_flood_profile[
            "flooded_critical_edge_share"
        ],
        (
            flooded_bridge_bottleneck_count
            + flooded_major_road_bottleneck_count
        ),
    )
)

prototype_scene_flood_profile_df[
    "scene_flood_burden"
] = scene_flood_burden

prototype_scene_flood_profile_df[
    "critical_network_disruption"
] = critical_disruption_class

# ------------------------------------------------------------
# 8. Build scene road–flood grounding token
# ------------------------------------------------------------

def yes_no(value):
    return "Yes" if value > 0 else "No"


prototype_scene_flood_token = " ".join(
    [
        (
            f"<SceneFloodBurden:"
            f"{scene_flood_burden}>"
        ),
        (
            f"<CriticalNetworkDisruption:"
            f"{critical_disruption_class}>"
        ),
        (
            f"<FloodedCriticalEdges:"
            f"{yes_no(flooded_critical_edge_count)}>"
        ),
        (
            f"<FloodedLowRedundancyEdges:"
            f"{yes_no(flooded_critical_low_redundancy_count)}>"
        ),
        (
            f"<FloodedPhysicalBridges:"
            f"{yes_no(flooded_physical_bridge_count)}>"
        ),
        (
            f"<FloodedBridgeBottlenecks:"
            f"{yes_no(flooded_bridge_bottleneck_count)}>"
        ),
        (
            f"<FloodedMajorRoadBottlenecks:"
            f"{yes_no(flooded_major_road_bottleneck_count)}>"
        ),
        (
            f"<RoadRasterCoverage:"
            f"{prototype_scene_flood_profile['raster_coverage_share']:.4f}>"
        ),
    ]
)

prototype_scene_flood_profile_df[
    "scene_road_flood_token"
] = prototype_scene_flood_token

# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print("\nPHYSICAL-EDGE FLOOD EXPOSURE")
print("-" * 72)

print(
    prototype_physical_edges_df[
        "road_flood_exposure_class"
    ]
    .value_counts(dropna=False)
    .to_string()
)

print("\nPROTOTYPE SCENE FLOOD PROFILE")
print("-" * 72)

display(
    prototype_scene_flood_profile_df.T
)

print("\nSCENE ROAD–FLOOD TOKEN")
print("-" * 72)

print(
    prototype_scene_flood_token
)

print("\nFLOODED CRITICAL PHYSICAL EDGES")
print("-" * 72)

critical_preview_columns = [
    column
    for column in [
        "physical_edge_id",
        "name",
        "highway",
        "hierarchy_group",
        "network_role",
        "event_flood_fraction",
        "road_flood_exposure_class",
        "is_critical_transport_edge",
        "is_critical_low_redundancy",
        "is_bridge_bottleneck",
        "is_major_road_bottleneck",
    ]
    if column in prototype_physical_edges_df.columns
]

flooded_critical_preview_df = (
    prototype_physical_edges_df.loc[
        prototype_physical_edges_df[
            "touches_event_flood"
        ].fillna(False)
    ]
    .sort_values(
        "event_flood_fraction",
        ascending=False,
    )
)

display(
    flooded_critical_preview_df[
        critical_preview_columns
    ].head(20)
)

print("=" * 72)

BUILDING PROTOTYPE ROAD–FLOOD SCENE SUMMARY

DIRECTED AND PHYSICAL EDGE COUNTS
------------------------------------------------------------------------
Directed road records : 902
Unique physical edges : 542

PHYSICAL-EDGE FLOOD EXPOSURE
------------------------------------------------------------------------
road_flood_exposure_class
No Detected Flood Exposure    422
Outside Raster Coverage        69
Low Flood Exposure             25
High Flood Exposure            15
Moderate Flood Exposure        11

PROTOTYPE SCENE FLOOD PROFILE
------------------------------------------------------------------------


,0
scene_id,Spain_8565131
directed_road_record_count,902
physical_edge_count,542
edges_inside_raster_count,473
edges_outside_raster_count,69
edges_with_valid_coverage_count,473
raster_coverage_share,0.8727
surface_water_edge_count,53
permanent_water_edge_count,0
event_flood_edge_count,53



SCENE ROAD–FLOOD TOKEN
------------------------------------------------------------------------
<SceneFloodBurden:Moderate> <CriticalNetworkDisruption:None> <FloodedCriticalEdges:No> <FloodedLowRedundancyEdges:No> <FloodedPhysicalBridges:No> <FloodedBridgeBottlenecks:No> <FloodedMajorRoadBottlenecks:No> <RoadRasterCoverage:0.8727>

FLOODED CRITICAL PHYSICAL EDGES
------------------------------------------------------------------------


,physical_edge_id,name,highway,hierarchy_group,event_flood_fraction,road_flood_exposure_class
382,2665217834__2665217835__0,None,unclassified,local,0.7090,High Flood Exposure
471,6138927629__6555102393__0,None,unclassified,local,0.6946,High Flood Exposure
483,6576753658__6576753662__0,None,residential,local,0.6757,High Flood Exposure
481,6576753657__6576753671__0,None,unclassified,local,0.6473,High Flood Exposure
266,1797999888__406556546__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.6049,High Flood Exposure
268,1797999888__6576753657__0,None,unclassified,local,0.5950,High Flood Exposure
469,6138927620__6576741024__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.5510,High Flood Exposure
310,2356968015__2356968071__0,None,unclassified,local,0.5389,High Flood Exposure
385,2665217845__2665217857__0,None,unclassified,local,0.4659,High Flood Exposure
448,406556529__6576741024__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.4561,High Flood Exposure


In [6]:
# ============================================================
# CELL 6: RESOLVE CRITICAL-ROAD FIELDS AND CORRECT SUMMARY
# ============================================================

import re
import numpy as np
import pandas as pd

print("=" * 72)
print("RESOLVING CRITICAL TRANSPORTATION FIELDS")
print("=" * 72)

# ------------------------------------------------------------
# 1. Inspect potentially relevant Notebook 05 columns
# ------------------------------------------------------------

search_terms = [
    "critical",
    "bridge",
    "bottleneck",
    "redund",
    "major",
    "vulnerab",
    "role",
    "hierarchy",
]

candidate_columns = [
    column
    for column in prototype_summary_gdf.columns
    if any(
        term in column.lower()
        for term in search_terms
    )
]

print("\nCANDIDATE TRANSPORTATION COLUMNS")
print("-" * 72)

for column in candidate_columns:
    series = prototype_summary_gdf[column]

    unique_preview = (
        series.dropna()
        .astype(str)
        .unique()[:8]
        .tolist()
    )

    print(f"\n{column}")
    print(f"  dtype  : {series.dtype}")
    print(f"  values : {unique_preview}")

# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------

def normalize_boolean_series(series):
    """
    Convert boolean, numeric, and common text values to Boolean.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        return (
            pd.to_numeric(series, errors="coerce")
            .fillna(0)
            .ne(0)
        )

    normalized = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    true_values = {
        "true",
        "yes",
        "y",
        "1",
        "critical",
        "high",
    }

    return normalized.isin(true_values)


def first_existing_column(
    dataframe,
    aliases,
):
    for alias in aliases:
        if alias in dataframe.columns:
            return alias

    return None


# ------------------------------------------------------------
# 3. Define possible aliases from Notebook 05
# ------------------------------------------------------------

field_aliases = {
    "is_critical_transport_edge": [
        "is_critical_transport_edge",
        "critical_transport_edge",
        "critical_edge",
        "is_critical_edge",
        "critical_transport_link",
        "is_critical_transport_link",
    ],

    "is_critical_low_redundancy": [
        "is_critical_low_redundancy",
        "critical_low_redundancy",
        "low_redundancy_critical",
        "critical_low_redundancy_edge",
        "is_low_redundancy_critical",
    ],

    "is_physical_bridge": [
        "is_physical_bridge",
        "physical_bridge",
        "bridge_flag",
        "is_bridge",
    ],

    "is_bridge_bottleneck": [
        "is_bridge_bottleneck",
        "bridge_bottleneck",
        "physical_bridge_bottleneck",
        "is_physical_bridge_bottleneck",
    ],

    "is_major_road": [
        "is_major_road",
        "major_road",
        "major_road_flag",
    ],

    "is_major_road_bottleneck": [
        "is_major_road_bottleneck",
        "major_road_bottleneck",
        "major_bottleneck",
        "is_major_bottleneck",
    ],

    "is_graph_bridge": [
        "is_graph_bridge",
        "graph_bridge",
        "network_bridge",
        "is_network_bridge",
    ],
}

resolved_columns = {}

print("\nRESOLVED FIELD ALIASES")
print("-" * 72)

for canonical_name, aliases in field_aliases.items():

    source_column = first_existing_column(
        prototype_summary_gdf,
        aliases,
    )

    resolved_columns[canonical_name] = source_column

    print(
        f"{canonical_name:<36}: "
        f"{source_column if source_column else 'NOT FOUND'}"
    )

# ------------------------------------------------------------
# 4. Create canonical Boolean fields
# ------------------------------------------------------------

corrected_prototype_gdf = (
    prototype_summary_gdf.copy()
)

for canonical_name, source_column in resolved_columns.items():

    if source_column is not None:
        corrected_prototype_gdf[canonical_name] = (
            normalize_boolean_series(
                corrected_prototype_gdf[
                    source_column
                ]
            )
        )
    else:
        corrected_prototype_gdf[canonical_name] = False

# ------------------------------------------------------------
# 5. Additional semantic fallback from network-role labels
# ------------------------------------------------------------

if "network_role" in corrected_prototype_gdf.columns:

    role_text = (
        corrected_prototype_gdf[
            "network_role"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    role_critical = role_text.str.contains(
        r"critical corridor|critical bridge",
        regex=True,
    )

    role_bridge = role_text.str.contains(
        r"critical bridge",
        regex=True,
    )

    role_single_access = role_text.str.contains(
        r"single access",
        regex=True,
    )

    # Apply only as a fallback/enrichment.
    corrected_prototype_gdf[
        "is_critical_transport_edge"
    ] = (
        corrected_prototype_gdf[
            "is_critical_transport_edge"
        ]
        | role_critical
    )

    corrected_prototype_gdf[
        "is_bridge_bottleneck"
    ] = (
        corrected_prototype_gdf[
            "is_bridge_bottleneck"
        ]
        | role_bridge
    )

    corrected_prototype_gdf[
        "is_critical_low_redundancy"
    ] = (
        corrected_prototype_gdf[
            "is_critical_low_redundancy"
        ]
        | (
            corrected_prototype_gdf[
                "is_critical_transport_edge"
            ]
            & role_single_access
        )
    )

# ------------------------------------------------------------
# 6. Rebuild physical-edge table
# ------------------------------------------------------------

corrected_aggregation_rules = {
    "intersects_raster": "max",
    "corridor_pixel_count": "max",
    "valid_pixel_count": "max",
    "invalid_pixel_count": "max",
    "surface_water_pixel_count": "max",
    "permanent_water_pixel_count": "max",
    "event_flood_pixel_count": "max",
    "non_water_pixel_count": "max",
    "valid_coverage_fraction": "max",
    "surface_water_fraction": "max",
    "permanent_water_fraction": "max",
    "event_flood_fraction": "max",
    "non_water_fraction": "max",
    "touches_surface_water": "max",
    "touches_permanent_water": "max",
    "touches_event_flood": "max",
    "road_flood_exposure_class": "first",
    "road_flood_token": "first",
}

for column in [
    "name",
    "highway",
    "hierarchy_group",
    "network_role",
    "vulnerability_label",
]:
    if column in corrected_prototype_gdf.columns:
        corrected_aggregation_rules[column] = "first"

for column in field_aliases:
    corrected_aggregation_rules[column] = "max"

for column in [
    "topological_vulnerability_score",
    "edge_criticality_score",
    "hierarchy_score",
    "local_edge_connectivity",
]:
    if column in corrected_prototype_gdf.columns:
        corrected_aggregation_rules[column] = "max"

corrected_physical_edges_df = (
    corrected_prototype_gdf
    .groupby(
        "physical_edge_id",
        as_index=False,
    )
    .agg(corrected_aggregation_rules)
)

# ------------------------------------------------------------
# 7. Recalculate critical-road metrics
# ------------------------------------------------------------

def count_true(dataframe, column):
    return int(
        dataframe[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )


def count_flooded_true(
    dataframe,
    column,
):
    return int(
        (
            dataframe[column]
            .fillna(False)
            .astype(bool)
            &
            dataframe["touches_event_flood"]
            .fillna(False)
            .astype(bool)
        ).sum()
    )


corrected_metrics = {
    "critical_edge_count": count_true(
        corrected_physical_edges_df,
        "is_critical_transport_edge",
    ),
    "flooded_critical_edge_count": count_flooded_true(
        corrected_physical_edges_df,
        "is_critical_transport_edge",
    ),

    "critical_low_redundancy_count": count_true(
        corrected_physical_edges_df,
        "is_critical_low_redundancy",
    ),
    "flooded_critical_low_redundancy_count": (
        count_flooded_true(
            corrected_physical_edges_df,
            "is_critical_low_redundancy",
        )
    ),

    "physical_bridge_count": count_true(
        corrected_physical_edges_df,
        "is_physical_bridge",
    ),
    "flooded_physical_bridge_count": count_flooded_true(
        corrected_physical_edges_df,
        "is_physical_bridge",
    ),

    "bridge_bottleneck_count": count_true(
        corrected_physical_edges_df,
        "is_bridge_bottleneck",
    ),
    "flooded_bridge_bottleneck_count": (
        count_flooded_true(
            corrected_physical_edges_df,
            "is_bridge_bottleneck",
        )
    ),

    "major_road_count": count_true(
        corrected_physical_edges_df,
        "is_major_road",
    ),
    "flooded_major_road_count": count_flooded_true(
        corrected_physical_edges_df,
        "is_major_road",
    ),

    "major_road_bottleneck_count": count_true(
        corrected_physical_edges_df,
        "is_major_road_bottleneck",
    ),
    "flooded_major_road_bottleneck_count": (
        count_flooded_true(
            corrected_physical_edges_df,
            "is_major_road_bottleneck",
        )
    ),
}

print("\nCORRECTED CRITICAL-INFRASTRUCTURE COUNTS")
print("-" * 72)

for metric_name, metric_value in corrected_metrics.items():
    print(
        f"{metric_name:<45}: "
        f"{metric_value:,}"
    )

# ------------------------------------------------------------
# 8. Validate against Notebook 05 scene profile
# ------------------------------------------------------------

prototype_transport_profile = (
    master_transportation_profiles_df.loc[
        master_transportation_profiles_df[
            "scene_id"
        ].astype(str)
        == PROTOTYPE_SCENE_ID
    ]
)

print("\nNOTEBOOK 05 PROFILE COMPARISON")
print("-" * 72)

comparison_terms = [
    "critical",
    "bridge",
    "major",
    "redund",
]

profile_comparison_columns = [
    column
    for column in prototype_transport_profile.columns
    if any(
        term in column.lower()
        for term in comparison_terms
    )
]

if not prototype_transport_profile.empty:
    for column in profile_comparison_columns:
        print(
            f"{column:<45}: "
            f"{prototype_transport_profile.iloc[0][column]}"
        )

# ------------------------------------------------------------
# 9. Correct flooded-critical preview
# ------------------------------------------------------------

flooded_critical_mask = (
    corrected_physical_edges_df[
        "touches_event_flood"
    ].fillna(False)
    &
    (
        corrected_physical_edges_df[
            "is_critical_transport_edge"
        ].fillna(False)
        |
        corrected_physical_edges_df[
            "is_critical_low_redundancy"
        ].fillna(False)
        |
        corrected_physical_edges_df[
            "is_bridge_bottleneck"
        ].fillna(False)
        |
        corrected_physical_edges_df[
            "is_major_road_bottleneck"
        ].fillna(False)
    )
)

corrected_flooded_critical_df = (
    corrected_physical_edges_df.loc[
        flooded_critical_mask
    ]
    .sort_values(
        "event_flood_fraction",
        ascending=False,
    )
    .copy()
)

corrected_preview_columns = [
    column
    for column in [
        "physical_edge_id",
        "name",
        "highway",
        "hierarchy_group",
        "network_role",
        "event_flood_fraction",
        "road_flood_exposure_class",
        "is_critical_transport_edge",
        "is_critical_low_redundancy",
        "is_physical_bridge",
        "is_bridge_bottleneck",
        "is_major_road",
        "is_major_road_bottleneck",
    ]
    if column in corrected_flooded_critical_df.columns
]

print("\nACTUAL FLOODED CRITICAL PHYSICAL EDGES")
print("-" * 72)

print(
    f"Records found: "
    f"{len(corrected_flooded_critical_df):,}"
)

display(
    corrected_flooded_critical_df[
        corrected_preview_columns
    ].head(25)
)

print("=" * 72)

RESOLVING CRITICAL TRANSPORTATION FIELDS

CANDIDATE TRANSPORTATION COLUMNS
------------------------------------------------------------------------

bridge
  dtype  : object
  values : ['yes']

is_bridge
  dtype  : bool
  values : ['True', 'False']

hierarchy_score
  dtype  : int64
  values : ['6', '1', '2', '5']

hierarchy_group
  dtype  : object
  values : ['limited_access', 'local', 'local_collector', 'major_arterial']

is_bridge_knowledge
  dtype  : bool
  values : ['True', 'False']

is_major_road
  dtype  : bool
  values : ['True', 'False']

is_critical_link
  dtype  : bool
  values : ['True', 'False']

touches_critical_node
  dtype  : bool
  values : ['True', 'False']

edge_criticality_score
  dtype  : float64
  values : ['0.7471325167037862', '0.7630150334075725', '0.6208101336302896', '0.3209957312546399', '0.3876623979213066', '0.4495220861172976', '0.4995220861172976', '0.4137435040831477']

critical_transport_edge
  dtype  : bool
  values : ['True', 'False']

is_network_brid

,physical_edge_id,name,highway,hierarchy_group,event_flood_fraction,road_flood_exposure_class,is_critical_transport_edge,is_critical_low_redundancy,is_physical_bridge,is_bridge_bottleneck,is_major_road,is_major_road_bottleneck
266,1797999888__406556546__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.6049,High Flood Exposure,True,False,False,False,False,False
268,1797999888__6576753657__0,None,unclassified,local,0.5950,High Flood Exposure,True,True,False,False,False,False
469,6138927620__6576741024__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.5510,High Flood Exposure,True,False,False,False,False,False
448,406556529__6576741024__0,Carretera de Crevillent a Catral,tertiary,local_collector,0.4561,High Flood Exposure,True,False,False,False,False,False
267,1797999888__6139088974__0,None,unclassified,local,0.4526,High Flood Exposure,True,False,False,False,False,False
30,1283202507__406556510__0,Camí del Conveni Nou,unclassified,local,0.4145,High Flood Exposure,True,True,False,False,False,False
468,6138927620__6576741023__0,None,residential,local,0.3095,High Flood Exposure,True,True,False,False,False,False
449,406556529__6576753587__0,None,residential,local,0.2885,Moderate Flood Exposure,True,True,False,False,False,False
478,6576741024__6576741025__0,None,residential,local,0.2593,Moderate Flood Exposure,True,True,False,False,False,False
334,2356983115__2954515765__0,None,unclassified,local,0.1366,Moderate Flood Exposure,True,False,True,False,False,False


In [7]:
# ============================================================
# CELL 7: BATCH ROAD–FLOOD GROUNDING FOR ALL SCENES
# ============================================================

from pathlib import Path
import json
import time
import traceback

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import rasterize
from shapely.geometry import box


print("=" * 80)
print("BATCH ROAD–FLOOD GROUNDING FOR ALL SCENES")
print("=" * 80)


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

LABEL_INVALID_VALUE = -1
LABEL_NOT_WATER_VALUE = 0
LABEL_WATER_VALUE = 1

JRC_NOT_PERMANENT_WATER_VALUE = 0
JRC_PERMANENT_WATER_VALUE = 1

ROAD_BUFFER_METERS = 10.0

FLOOD_TOUCH_THRESHOLD = 0.01
MODERATE_FLOOD_THRESHOLD = 0.10
HIGH_FLOOD_THRESHOLD = 0.30


# ------------------------------------------------------------
# 2. Output directories
# ------------------------------------------------------------

BATCH_OUTPUT_DIR = (
    ROAD_FLOOD_OUTPUT_DIR
    / "batch_grounding"
)

DIRECTED_ROAD_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "directed_road_grounding"
)

PHYSICAL_EDGE_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "physical_edge_grounding"
)

SCENE_PROFILE_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "scene_profiles"
)

SCENE_TOKEN_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "scene_tokens"
)

MASTER_OUTPUT_DIR = (
    BATCH_OUTPUT_DIR
    / "master"
)

for directory in [
    BATCH_OUTPUT_DIR,
    DIRECTED_ROAD_OUTPUT_DIR,
    PHYSICAL_EDGE_OUTPUT_DIR,
    SCENE_PROFILE_OUTPUT_DIR,
    SCENE_TOKEN_OUTPUT_DIR,
    MASTER_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


print("\nOUTPUT DIRECTORIES")
print("-" * 80)

print(
    f"Batch root            : "
    f"{BATCH_OUTPUT_DIR}"
)

print(
    f"Directed-road outputs : "
    f"{DIRECTED_ROAD_OUTPUT_DIR}"
)

print(
    f"Physical-edge outputs : "
    f"{PHYSICAL_EDGE_OUTPUT_DIR}"
)

print(
    f"Scene profiles        : "
    f"{SCENE_PROFILE_OUTPUT_DIR}"
)

print(
    f"Scene tokens          : "
    f"{SCENE_TOKEN_OUTPUT_DIR}"
)

print(
    f"Master outputs        : "
    f"{MASTER_OUTPUT_DIR}"
)


# ------------------------------------------------------------
# 3. Validate grounding inventory
# ------------------------------------------------------------

required_inventory_columns = [
    "scene_id",
    "road_knowledge_path",
    "label_path",
    "jrc_water_path",
    "grounding_ready",
    "grounding_ready_with_jrc",
]

missing_inventory_columns = [
    column
    for column in required_inventory_columns
    if column not in grounding_inventory_df.columns
]

if missing_inventory_columns:
    raise ValueError(
        "Grounding inventory is missing required columns: "
        f"{missing_inventory_columns}"
    )

batch_inventory_df = (
    grounding_inventory_df.loc[
        grounding_inventory_df[
            "grounding_ready_with_jrc"
        ].fillna(False)
    ]
    .copy()
    .sort_values("scene_id")
    .reset_index(drop=True)
)

print("\nBATCH INVENTORY")
print("-" * 80)

print(
    f"Scenes available       : "
    f"{len(grounding_inventory_df):,}"
)

print(
    f"Scenes selected        : "
    f"{len(batch_inventory_df):,}"
)

if batch_inventory_df.empty:
    raise ValueError(
        "No scenes are ready for road–flood grounding."
    )


# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------

def normalize_boolean_series(series):
    """
    Convert Boolean, numeric, and common string values to Boolean.
    """

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)

    if pd.api.types.is_numeric_dtype(series):
        return (
            pd.to_numeric(
                series,
                errors="coerce",
            )
            .fillna(0)
            .ne(0)
        )

    normalized = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return normalized.isin(
        {
            "true",
            "yes",
            "y",
            "1",
        }
    )


def ensure_boolean_column(
    dataframe,
    source_column,
    output_column,
):
    """
    Create a canonical Boolean column from an existing field.
    """

    if source_column in dataframe.columns:
        dataframe[output_column] = (
            normalize_boolean_series(
                dataframe[source_column]
            )
        )
    else:
        dataframe[output_column] = False

    return dataframe


def build_physical_edge_id(row):
    """
    Give u->v and v->u records the same physical-edge ID.
    """

    u_value = str(
        row.get(
            "u",
            "",
        )
    )

    v_value = str(
        row.get(
            "v",
            "",
        )
    )

    endpoint_1, endpoint_2 = sorted(
        [
            u_value,
            v_value,
        ]
    )

    key_value = str(
        row.get(
            "key",
            0,
        )
    )

    return (
        f"{endpoint_1}__"
        f"{endpoint_2}__"
        f"{key_value}"
    )


def classify_road_flood_exposure(
    event_flood_fraction,
    valid_pixel_count,
):
    """
    Classify the flood exposure of a buffered road corridor.
    """

    if valid_pixel_count == 0:
        return "No Valid Raster Coverage"

    if event_flood_fraction < FLOOD_TOUCH_THRESHOLD:
        return "No Detected Flood Exposure"

    if event_flood_fraction < MODERATE_FLOOD_THRESHOLD:
        return "Low Flood Exposure"

    if event_flood_fraction < HIGH_FLOOD_THRESHOLD:
        return "Moderate Flood Exposure"

    return "High Flood Exposure"


def classify_scene_flood_burden(
    flood_edge_share,
):
    """
    Classify the scene-level share of physical roads exposed.
    """

    if flood_edge_share < 0.01:
        return "Minimal"

    if flood_edge_share < 0.10:
        return "Low"

    if flood_edge_share < 0.25:
        return "Moderate"

    return "High"


def classify_critical_disruption(
    flooded_critical_share,
    flooded_bottleneck_count,
):
    """
    Classify disruption to critical or low-redundancy links.
    """

    if (
        flooded_critical_share == 0
        and flooded_bottleneck_count == 0
    ):
        return "None"

    if (
        flooded_critical_share < 0.10
        and flooded_bottleneck_count == 0
    ):
        return "Low"

    if (
        flooded_critical_share < 0.30
        and flooded_bottleneck_count <= 2
    ):
        return "Moderate"

    return "High"


def safe_share(
    numerator,
    denominator,
):
    """
    Divide safely and round to four decimal places.
    """

    if denominator == 0:
        return 0.0

    return round(
        float(numerator) / float(denominator),
        4,
    )


def safe_mean(series):
    """
    Return a finite rounded mean or zero.
    """

    numeric_series = pd.to_numeric(
        series,
        errors="coerce",
    )

    value = numeric_series.mean()

    if pd.isna(value):
        return 0.0

    return round(
        float(value),
        4,
    )


def safe_max(series):
    """
    Return a finite rounded maximum or zero.
    """

    numeric_series = pd.to_numeric(
        series,
        errors="coerce",
    )

    value = numeric_series.max()

    if pd.isna(value):
        return 0.0

    return round(
        float(value),
        4,
    )


def count_true(
    dataframe,
    column,
):
    """
    Count true values in a Boolean-like column.
    """

    if column not in dataframe.columns:
        return 0

    return int(
        dataframe[column]
        .fillna(False)
        .astype(bool)
        .sum()
    )


def count_flooded_true(
    dataframe,
    column,
):
    """
    Count flooded physical edges belonging to a road category.
    """

    if column not in dataframe.columns:
        return 0

    return int(
        (
            dataframe[column]
            .fillna(False)
            .astype(bool)
            &
            dataframe[
                "touches_event_flood"
            ]
            .fillna(False)
            .astype(bool)
        ).sum()
    )


def build_road_flood_token(row):
    """
    Build a compact semantic token for one directed road record.
    """

    exposure_class = (
        str(
            row.get(
                "road_flood_exposure_class",
                "Unknown",
            )
        )
        .replace(
            " ",
            "",
        )
    )

    surface_water_value = (
        "Yes"
        if bool(
            row.get(
                "touches_surface_water",
                False,
            )
        )
        else "No"
    )

    permanent_water_value = (
        "Yes"
        if bool(
            row.get(
                "touches_permanent_water",
                False,
            )
        )
        else "No"
    )

    event_flood_value = (
        "Yes"
        if bool(
            row.get(
                "touches_event_flood",
                False,
            )
        )
        else "No"
    )

    critical_value = (
        "Yes"
        if bool(
            row.get(
                "is_critical_transport_edge",
                False,
            )
        )
        else "No"
    )

    low_redundancy_value = (
        "Yes"
        if bool(
            row.get(
                "is_critical_low_redundancy",
                False,
            )
        )
        else "No"
    )

    bridge_value = (
        "Yes"
        if bool(
            row.get(
                "is_physical_bridge",
                False,
            )
        )
        else "No"
    )

    return " ".join(
        [
            (
                f"<RoadFloodExposure:"
                f"{exposure_class}>"
            ),
            (
                f"<TouchesSurfaceWater:"
                f"{surface_water_value}>"
            ),
            (
                f"<TouchesPermanentWater:"
                f"{permanent_water_value}>"
            ),
            (
                f"<TouchesEventFlood:"
                f"{event_flood_value}>"
            ),
            (
                f"<CriticalTransportEdge:"
                f"{critical_value}>"
            ),
            (
                f"<CriticalLowRedundancy:"
                f"{low_redundancy_value}>"
            ),
            (
                f"<PhysicalBridge:"
                f"{bridge_value}>"
            ),
        ]
    )


def build_scene_road_flood_token(
    scene_flood_burden,
    critical_disruption_class,
    scene_profile,
):
    """
    Build the scene-level road–flood grounding token.
    """

    def yes_no(value):
        return (
            "Yes"
            if int(value) > 0
            else "No"
        )

    return " ".join(
        [
            (
                f"<SceneFloodBurden:"
                f"{scene_flood_burden}>"
            ),
            (
                f"<CriticalNetworkDisruption:"
                f"{critical_disruption_class}>"
            ),
            (
                f"<FloodedCriticalEdges:"
                f"{yes_no(scene_profile['flooded_critical_edge_count'])}>"
            ),
            (
                f"<FloodedLowRedundancyEdges:"
                f"{yes_no(scene_profile['flooded_critical_low_redundancy_count'])}>"
            ),
            (
                f"<FloodedPhysicalBridges:"
                f"{yes_no(scene_profile['flooded_physical_bridge_count'])}>"
            ),
            (
                f"<FloodedBridgeBottlenecks:"
                f"{yes_no(scene_profile['flooded_bridge_bottleneck_count'])}>"
            ),
            (
                f"<FloodedMajorRoadBottlenecks:"
                f"{yes_no(scene_profile['flooded_major_road_bottleneck_count'])}>"
            ),
            (
                f"<RoadRasterCoverage:"
                f"{scene_profile['raster_coverage_share']:.4f}>"
            ),
            (
                f"<FloodedRoadShare:"
                f"{scene_profile['event_flood_edge_share']:.4f}>"
            ),
        ]
    )


def make_json_safe(value):
    """
    Convert NumPy and pandas values to JSON-compatible values.
    """

    if value is None:
        return None

    if isinstance(
        value,
        (
            np.integer,
        ),
    ):
        return int(value)

    if isinstance(
        value,
        (
            np.floating,
        ),
    ):
        if np.isnan(value):
            return None

        return float(value)

    if isinstance(
        value,
        (
            np.bool_,
        ),
    ):
        return bool(value)

    if isinstance(
        value,
        pd.Timestamp,
    ):
        return value.isoformat()

    if pd.isna(value):
        return None

    return value


def clean_dataframe_for_json(dataframe):
    """
    Convert a DataFrame into JSON-safe records.
    """

    records = dataframe.to_dict(
        orient="records"
    )

    return [
        {
            key: make_json_safe(value)
            for key, value in record.items()
        }
        for record in records
    ]


# ------------------------------------------------------------
# 5. Main single-scene grounding function
# ------------------------------------------------------------

def process_scene_road_flood_grounding(
    inventory_row,
):
    """
    Process one scene and return:
      - directed road GeoDataFrame
      - physical-edge DataFrame
      - scene profile dictionary
      - scene token dictionary
      - processing record dictionary
    """

    scene_start_time = time.time()

    scene_id = str(
        inventory_row["scene_id"]
    )

    road_path = Path(
        inventory_row[
            "road_knowledge_path"
        ]
    )

    label_path = Path(
        inventory_row[
            "label_path"
        ]
    )

    jrc_path = Path(
        inventory_row[
            "jrc_water_path"
        ]
    )

    # --------------------------------------------------------
    # Load raster data
    # --------------------------------------------------------

    with rasterio.open(
        label_path
    ) as label_src:

        label_array = label_src.read(1)

        raster_transform = (
            label_src.transform
        )

        raster_crs = label_src.crs
        raster_width = label_src.width
        raster_height = label_src.height
        raster_bounds = label_src.bounds

    with rasterio.open(
        jrc_path
    ) as jrc_src:

        jrc_array = jrc_src.read(1)

        if (
            jrc_src.width != raster_width
            or jrc_src.height != raster_height
        ):
            raise ValueError(
                f"{scene_id}: Label and JRC dimensions differ."
            )

        if jrc_src.crs != raster_crs:
            raise ValueError(
                f"{scene_id}: Label and JRC CRS differ."
            )

        if not np.allclose(
            np.array(jrc_src.bounds),
            np.array(raster_bounds),
        ):
            raise ValueError(
                f"{scene_id}: Label and JRC bounds differ."
            )

    # --------------------------------------------------------
    # Construct semantic masks
    # --------------------------------------------------------

    valid_mask = (
        label_array
        != LABEL_INVALID_VALUE
    )

    surface_water_mask = (
        (
            label_array
            == LABEL_WATER_VALUE
        )
        & valid_mask
    )

    permanent_water_mask = (
        (
            jrc_array
            == JRC_PERMANENT_WATER_VALUE
        )
        & valid_mask
    )

    event_flood_proxy_mask = (
        surface_water_mask
        & ~permanent_water_mask
    )

    non_water_mask = (
        (
            label_array
            == LABEL_NOT_WATER_VALUE
        )
        & valid_mask
    )

    # --------------------------------------------------------
    # Load and prepare roads
    # --------------------------------------------------------

    roads_gdf = gpd.read_file(
        road_path,
    )

    roads_gdf = roads_gdf.loc[
        roads_gdf.geometry.notna()
        & ~roads_gdf.geometry.is_empty
    ].copy()

    if roads_gdf.empty:
        raise ValueError(
            f"{scene_id}: Road layer is empty."
        )

    if roads_gdf.crs is None:
        raise ValueError(
            f"{scene_id}: Road layer has no CRS."
        )

    # Exact Notebook 05 fields.
    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "critical_transport_edge",
        "is_critical_transport_edge",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "critical_low_redundancy_edge",
        "is_critical_low_redundancy",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "is_bridge",
        "is_physical_bridge",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "bridge_bottleneck_edge",
        "is_bridge_bottleneck",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "is_major_road",
        "is_major_road",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "major_road_bottleneck_edge",
        "is_major_road_bottleneck",
    )

    roads_gdf = ensure_boolean_column(
        roads_gdf,
        "is_network_bridge_edge",
        "is_graph_bridge",
    )

    roads_gdf[
        "physical_edge_id"
    ] = roads_gdf.apply(
        build_physical_edge_id,
        axis=1,
    )

    # --------------------------------------------------------
    # Project and buffer roads
    # --------------------------------------------------------

    projected_crs = (
        roads_gdf.estimate_utm_crs()
    )

    if projected_crs is None:
        raise ValueError(
            f"{scene_id}: Could not estimate projected CRS."
        )

    roads_projected_gdf = (
        roads_gdf.to_crs(
            projected_crs
        )
    )

    buffered_roads_gdf = (
        roads_projected_gdf.copy()
    )

    buffered_roads_gdf.geometry = (
        buffered_roads_gdf.geometry.buffer(
            ROAD_BUFFER_METERS
        )
    )

    buffered_roads_gdf = (
        buffered_roads_gdf.to_crs(
            raster_crs
        )
    )

    # --------------------------------------------------------
    # Clip roads to raster extent
    # --------------------------------------------------------

    raster_extent_geometry = box(
        raster_bounds.left,
        raster_bounds.bottom,
        raster_bounds.right,
        raster_bounds.top,
    )

    buffered_roads_gdf[
        "intersects_raster"
    ] = buffered_roads_gdf.geometry.intersects(
        raster_extent_geometry
    )

    buffered_roads_gdf.geometry = (
        buffered_roads_gdf.geometry.intersection(
            raster_extent_geometry
        )
    )

    # --------------------------------------------------------
    # Raster grounding for each directed road record
    # --------------------------------------------------------

    road_flood_records = []

    for road_index, road_row in (
        buffered_roads_gdf.iterrows()
    ):

        road_geometry = road_row.geometry

        record = {
            "road_index": road_index,
            "intersects_raster": bool(
                road_row[
                    "intersects_raster"
                ]
            ),
            "corridor_pixel_count": 0,
            "valid_pixel_count": 0,
            "invalid_pixel_count": 0,
            "surface_water_pixel_count": 0,
            "permanent_water_pixel_count": 0,
            "event_flood_pixel_count": 0,
            "non_water_pixel_count": 0,
            "valid_coverage_fraction": 0.0,
            "surface_water_fraction": 0.0,
            "permanent_water_fraction": 0.0,
            "event_flood_fraction": 0.0,
            "non_water_fraction": 0.0,
            "touches_surface_water": False,
            "touches_permanent_water": False,
            "touches_event_flood": False,
        }

        if (
            road_geometry is None
            or road_geometry.is_empty
            or not record[
                "intersects_raster"
            ]
        ):
            record[
                "road_flood_exposure_class"
            ] = "Outside Raster Coverage"

            road_flood_records.append(
                record
            )

            continue

        road_corridor_mask = rasterize(
            [
                (
                    road_geometry,
                    1,
                )
            ],
            out_shape=(
                raster_height,
                raster_width,
            ),
            transform=raster_transform,
            fill=0,
            all_touched=True,
            dtype="uint8",
        ).astype(bool)

        corridor_pixel_count = int(
            road_corridor_mask.sum()
        )

        valid_road_pixels = (
            road_corridor_mask
            & valid_mask
        )

        invalid_road_pixels = (
            road_corridor_mask
            & ~valid_mask
        )

        surface_water_road_pixels = (
            road_corridor_mask
            & surface_water_mask
        )

        permanent_water_road_pixels = (
            road_corridor_mask
            & permanent_water_mask
        )

        event_flood_road_pixels = (
            road_corridor_mask
            & event_flood_proxy_mask
        )

        non_water_road_pixels = (
            road_corridor_mask
            & non_water_mask
        )

        valid_pixel_count = int(
            valid_road_pixels.sum()
        )

        invalid_pixel_count = int(
            invalid_road_pixels.sum()
        )

        surface_water_pixel_count = int(
            surface_water_road_pixels.sum()
        )

        permanent_water_pixel_count = int(
            permanent_water_road_pixels.sum()
        )

        event_flood_pixel_count = int(
            event_flood_road_pixels.sum()
        )

        non_water_pixel_count = int(
            non_water_road_pixels.sum()
        )

        valid_coverage_fraction = safe_share(
            valid_pixel_count,
            corridor_pixel_count,
        )

        surface_water_fraction = safe_share(
            surface_water_pixel_count,
            valid_pixel_count,
        )

        permanent_water_fraction = safe_share(
            permanent_water_pixel_count,
            valid_pixel_count,
        )

        event_flood_fraction = safe_share(
            event_flood_pixel_count,
            valid_pixel_count,
        )

        non_water_fraction = safe_share(
            non_water_pixel_count,
            valid_pixel_count,
        )

        record.update(
            {
                "corridor_pixel_count": (
                    corridor_pixel_count
                ),
                "valid_pixel_count": (
                    valid_pixel_count
                ),
                "invalid_pixel_count": (
                    invalid_pixel_count
                ),
                "surface_water_pixel_count": (
                    surface_water_pixel_count
                ),
                "permanent_water_pixel_count": (
                    permanent_water_pixel_count
                ),
                "event_flood_pixel_count": (
                    event_flood_pixel_count
                ),
                "non_water_pixel_count": (
                    non_water_pixel_count
                ),
                "valid_coverage_fraction": (
                    valid_coverage_fraction
                ),
                "surface_water_fraction": (
                    surface_water_fraction
                ),
                "permanent_water_fraction": (
                    permanent_water_fraction
                ),
                "event_flood_fraction": (
                    event_flood_fraction
                ),
                "non_water_fraction": (
                    non_water_fraction
                ),
                "touches_surface_water": (
                    surface_water_pixel_count > 0
                ),
                "touches_permanent_water": (
                    permanent_water_pixel_count > 0
                ),
                "touches_event_flood": (
                    event_flood_pixel_count > 0
                ),
                "road_flood_exposure_class": (
                    classify_road_flood_exposure(
                        event_flood_fraction,
                        valid_pixel_count,
                    )
                ),
            }
        )

        road_flood_records.append(
            record
        )

    road_flood_metrics_df = (
        pd.DataFrame(
            road_flood_records
        )
        .set_index(
            "road_index"
        )
    )

    directed_grounding_gdf = (
        roads_gdf.join(
            road_flood_metrics_df,
            how="left",
        )
    )

    directed_grounding_gdf[
        "scene_id"
    ] = scene_id

    directed_grounding_gdf[
        "road_flood_token"
    ] = directed_grounding_gdf.apply(
        build_road_flood_token,
        axis=1,
    )

    # --------------------------------------------------------
    # Collapse directed records to physical road edges
    # --------------------------------------------------------

    aggregation_rules = {
        "scene_id": "first",
        "intersects_raster": "max",
        "corridor_pixel_count": "max",
        "valid_pixel_count": "max",
        "invalid_pixel_count": "max",
        "surface_water_pixel_count": "max",
        "permanent_water_pixel_count": "max",
        "event_flood_pixel_count": "max",
        "non_water_pixel_count": "max",
        "valid_coverage_fraction": "max",
        "surface_water_fraction": "max",
        "permanent_water_fraction": "max",
        "event_flood_fraction": "max",
        "non_water_fraction": "max",
        "touches_surface_water": "max",
        "touches_permanent_water": "max",
        "touches_event_flood": "max",
        "road_flood_exposure_class": "first",
        "road_flood_token": "first",
        "is_critical_transport_edge": "max",
        "is_critical_low_redundancy": "max",
        "is_physical_bridge": "max",
        "is_bridge_bottleneck": "max",
        "is_major_road": "max",
        "is_major_road_bottleneck": "max",
        "is_graph_bridge": "max",
    }

    optional_first_columns = [
        "u",
        "v",
        "key",
        "osmid",
        "name",
        "highway",
        "hierarchy_group",
        "HierarchyLabel",
        "NetworkRole",
        "RedundancyLabel",
        "VulnerabilityLabel",
    ]

    optional_max_columns = [
        "hierarchy_score",
        "edge_criticality_score",
        "topological_vulnerability_score",
        "high_topological_vulnerability",
        "local_edge_connectivity",
        "touches_critical_node",
        "is_critical_link",
        "is_bridge_knowledge",
    ]

    for column in optional_first_columns:
        if column in directed_grounding_gdf.columns:
            aggregation_rules[
                column
            ] = "first"

    for column in optional_max_columns:
        if column in directed_grounding_gdf.columns:
            aggregation_rules[
                column
            ] = "max"

    physical_edges_df = (
        directed_grounding_gdf
        .drop(
            columns="geometry"
        )
        .groupby(
            "physical_edge_id",
            as_index=False,
        )
        .agg(
            aggregation_rules
        )
    )

    # --------------------------------------------------------
    # Scene-level general metrics
    # --------------------------------------------------------

    total_physical_edges = int(
        len(
            physical_edges_df
        )
    )

    edges_inside_raster = count_true(
        physical_edges_df,
        "intersects_raster",
    )

    edges_outside_raster = int(
        total_physical_edges
        - edges_inside_raster
    )

    edges_with_valid_coverage = int(
        (
            physical_edges_df[
                "valid_pixel_count"
            ] > 0
        ).sum()
    )

    surface_water_edges = count_true(
        physical_edges_df,
        "touches_surface_water",
    )

    permanent_water_edges = count_true(
        physical_edges_df,
        "touches_permanent_water",
    )

    event_flood_edges = count_true(
        physical_edges_df,
        "touches_event_flood",
    )

    low_flood_edges = int(
        (
            physical_edges_df[
                "road_flood_exposure_class"
            ]
            == "Low Flood Exposure"
        ).sum()
    )

    moderate_flood_edges = int(
        (
            physical_edges_df[
                "road_flood_exposure_class"
            ]
            == "Moderate Flood Exposure"
        ).sum()
    )

    high_flood_edges = int(
        (
            physical_edges_df[
                "road_flood_exposure_class"
            ]
            == "High Flood Exposure"
        ).sum()
    )

    no_detected_flood_edges = int(
        (
            physical_edges_df[
                "road_flood_exposure_class"
            ]
            == "No Detected Flood Exposure"
        ).sum()
    )

    # --------------------------------------------------------
    # Scene-level critical-infrastructure metrics
    # --------------------------------------------------------

    critical_edge_count = count_true(
        physical_edges_df,
        "is_critical_transport_edge",
    )

    flooded_critical_edge_count = (
        count_flooded_true(
            physical_edges_df,
            "is_critical_transport_edge",
        )
    )

    critical_low_redundancy_count = (
        count_true(
            physical_edges_df,
            "is_critical_low_redundancy",
        )
    )

    flooded_critical_low_redundancy_count = (
        count_flooded_true(
            physical_edges_df,
            "is_critical_low_redundancy",
        )
    )

    physical_bridge_count = count_true(
        physical_edges_df,
        "is_physical_bridge",
    )

    flooded_physical_bridge_count = (
        count_flooded_true(
            physical_edges_df,
            "is_physical_bridge",
        )
    )

    bridge_bottleneck_count = count_true(
        physical_edges_df,
        "is_bridge_bottleneck",
    )

    flooded_bridge_bottleneck_count = (
        count_flooded_true(
            physical_edges_df,
            "is_bridge_bottleneck",
        )
    )

    major_road_count = count_true(
        physical_edges_df,
        "is_major_road",
    )

    flooded_major_road_count = (
        count_flooded_true(
            physical_edges_df,
            "is_major_road",
        )
    )

    major_road_bottleneck_count = (
        count_true(
            physical_edges_df,
            "is_major_road_bottleneck",
        )
    )

    flooded_major_road_bottleneck_count = (
        count_flooded_true(
            physical_edges_df,
            "is_major_road_bottleneck",
        )
    )

    graph_bridge_count = count_true(
        physical_edges_df,
        "is_graph_bridge",
    )

    flooded_graph_bridge_count = (
        count_flooded_true(
            physical_edges_df,
            "is_graph_bridge",
        )
    )

    # --------------------------------------------------------
    # Scene-level pixel metrics
    # --------------------------------------------------------

    valid_pixel_count = int(
        valid_mask.sum()
    )

    invalid_pixel_count = int(
        (~valid_mask).sum()
    )

    surface_water_pixel_count = int(
        surface_water_mask.sum()
    )

    permanent_water_pixel_count = int(
        permanent_water_mask.sum()
    )

    event_flood_pixel_count = int(
        event_flood_proxy_mask.sum()
    )

    non_water_pixel_count = int(
        non_water_mask.sum()
    )

    jrc_outside_label_water_count = int(
        (
            permanent_water_mask
            & ~surface_water_mask
        ).sum()
    )

    # --------------------------------------------------------
    # Scene profile
    # --------------------------------------------------------

    event_flood_edge_share = safe_share(
        event_flood_edges,
        edges_with_valid_coverage,
    )

    flooded_critical_edge_share = safe_share(
        flooded_critical_edge_count,
        critical_edge_count,
    )

    flooded_low_redundancy_share = safe_share(
        flooded_critical_low_redundancy_count,
        critical_low_redundancy_count,
    )

    raster_coverage_share = safe_share(
        edges_inside_raster,
        total_physical_edges,
    )

    flood_bottleneck_count = int(
        flooded_bridge_bottleneck_count
        + flooded_major_road_bottleneck_count
    )

    scene_flood_burden = (
        classify_scene_flood_burden(
            event_flood_edge_share
        )
    )

    critical_disruption_class = (
        classify_critical_disruption(
            flooded_critical_edge_share,
            flood_bottleneck_count,
        )
    )

    valid_physical_edges_df = (
        physical_edges_df.loc[
            physical_edges_df[
                "valid_pixel_count"
            ] > 0
        ]
    )

    scene_runtime_seconds = round(
        time.time()
        - scene_start_time,
        2,
    )

    scene_profile = {
        "scene_id": scene_id,

        # Input paths
        "road_knowledge_path": str(
            road_path
        ),
        "label_path": str(
            label_path
        ),
        "jrc_water_path": str(
            jrc_path
        ),

        # Raster information
        "raster_crs": str(
            raster_crs
        ),
        "raster_width": int(
            raster_width
        ),
        "raster_height": int(
            raster_height
        ),
        "valid_pixel_count": (
            valid_pixel_count
        ),
        "invalid_pixel_count": (
            invalid_pixel_count
        ),
        "surface_water_pixel_count": (
            surface_water_pixel_count
        ),
        "permanent_water_pixel_count": (
            permanent_water_pixel_count
        ),
        "event_flood_pixel_count": (
            event_flood_pixel_count
        ),
        "non_water_pixel_count": (
            non_water_pixel_count
        ),
        "jrc_outside_label_water_count": (
            jrc_outside_label_water_count
        ),
        "surface_water_pixel_share": (
            safe_share(
                surface_water_pixel_count,
                valid_pixel_count,
            )
        ),
        "event_flood_pixel_share": (
            safe_share(
                event_flood_pixel_count,
                valid_pixel_count,
            )
        ),

        # Record structure
        "directed_road_record_count": int(
            len(
                directed_grounding_gdf
            )
        ),
        "physical_edge_count": (
            total_physical_edges
        ),

        # Raster-road coverage
        "edges_inside_raster_count": (
            edges_inside_raster
        ),
        "edges_outside_raster_count": (
            edges_outside_raster
        ),
        "edges_with_valid_coverage_count": (
            edges_with_valid_coverage
        ),
        "raster_coverage_share": (
            raster_coverage_share
        ),

        # General road flood exposure
        "surface_water_edge_count": (
            surface_water_edges
        ),
        "permanent_water_edge_count": (
            permanent_water_edges
        ),
        "event_flood_edge_count": (
            event_flood_edges
        ),
        "event_flood_edge_share": (
            event_flood_edge_share
        ),
        "no_detected_flood_edge_count": (
            no_detected_flood_edges
        ),
        "low_flood_exposure_edge_count": (
            low_flood_edges
        ),
        "moderate_flood_exposure_edge_count": (
            moderate_flood_edges
        ),
        "high_flood_exposure_edge_count": (
            high_flood_edges
        ),
        "mean_event_flood_fraction": (
            safe_mean(
                valid_physical_edges_df[
                    "event_flood_fraction"
                ]
            )
        ),
        "maximum_event_flood_fraction": (
            safe_max(
                physical_edges_df[
                    "event_flood_fraction"
                ]
            )
        ),

        # Critical transportation exposure
        "critical_edge_count": (
            critical_edge_count
        ),
        "flooded_critical_edge_count": (
            flooded_critical_edge_count
        ),
        "flooded_critical_edge_share": (
            flooded_critical_edge_share
        ),

        "critical_low_redundancy_count": (
            critical_low_redundancy_count
        ),
        "flooded_critical_low_redundancy_count": (
            flooded_critical_low_redundancy_count
        ),
        "flooded_critical_low_redundancy_share": (
            flooded_low_redundancy_share
        ),

        "physical_bridge_count": (
            physical_bridge_count
        ),
        "flooded_physical_bridge_count": (
            flooded_physical_bridge_count
        ),

        "graph_bridge_count": (
            graph_bridge_count
        ),
        "flooded_graph_bridge_count": (
            flooded_graph_bridge_count
        ),

        "bridge_bottleneck_count": (
            bridge_bottleneck_count
        ),
        "flooded_bridge_bottleneck_count": (
            flooded_bridge_bottleneck_count
        ),

        "major_road_count": (
            major_road_count
        ),
        "flooded_major_road_count": (
            flooded_major_road_count
        ),

        "major_road_bottleneck_count": (
            major_road_bottleneck_count
        ),
        "flooded_major_road_bottleneck_count": (
            flooded_major_road_bottleneck_count
        ),

        # Semantic classifications
        "scene_flood_burden": (
            scene_flood_burden
        ),
        "critical_network_disruption": (
            critical_disruption_class
        ),

        # Processing metadata
        "road_buffer_meters": float(
            ROAD_BUFFER_METERS
        ),
        "processing_runtime_seconds": (
            scene_runtime_seconds
        ),
    }

    scene_token = (
        build_scene_road_flood_token(
            scene_flood_burden,
            critical_disruption_class,
            scene_profile,
        )
    )

    scene_profile[
        "scene_road_flood_token"
    ] = scene_token

    scene_token_record = {
        "scene_id": scene_id,
        "scene_flood_burden": (
            scene_flood_burden
        ),
        "critical_network_disruption": (
            critical_disruption_class
        ),
        "scene_road_flood_token": (
            scene_token
        ),
    }

    processing_record = {
        "scene_id": scene_id,
        "status": "success",
        "directed_road_record_count": int(
            len(
                directed_grounding_gdf
            )
        ),
        "physical_edge_count": (
            total_physical_edges
        ),
        "event_flood_edge_count": (
            event_flood_edges
        ),
        "flooded_critical_edge_count": (
            flooded_critical_edge_count
        ),
        "runtime_seconds": (
            scene_runtime_seconds
        ),
        "error_message": None,
    }

    return {
        "directed_grounding_gdf": (
            directed_grounding_gdf
        ),
        "physical_edges_df": (
            physical_edges_df
        ),
        "scene_profile": (
            scene_profile
        ),
        "scene_token_record": (
            scene_token_record
        ),
        "processing_record": (
            processing_record
        ),
    }


# ------------------------------------------------------------
# 6. Run all scenes
# ------------------------------------------------------------

all_scene_profiles = []
all_scene_tokens = []
all_processing_records = []
all_physical_edge_tables = []

batch_start_time = time.time()

total_scenes = len(
    batch_inventory_df
)

print("\nPROCESSING SCENES")
print("-" * 80)


for scene_number, inventory_row in (
    batch_inventory_df.iterrows()
):

    scene_id = str(
        inventory_row[
            "scene_id"
        ]
    )

    display_number = (
        scene_number + 1
    )

    print(
        f"\n[{display_number:02d}/"
        f"{total_scenes:02d}] "
        f"Processing {scene_id}"
    )

    try:
        result = (
            process_scene_road_flood_grounding(
                inventory_row
            )
        )

        directed_grounding_gdf = (
            result[
                "directed_grounding_gdf"
            ]
        )

        physical_edges_df = (
            result[
                "physical_edges_df"
            ]
        )

        scene_profile = (
            result[
                "scene_profile"
            ]
        )

        scene_token_record = (
            result[
                "scene_token_record"
            ]
        )

        processing_record = (
            result[
                "processing_record"
            ]
        )

        # ----------------------------------------------------
        # Save directed road grounding
        # ----------------------------------------------------

        directed_gpkg_path = (
            DIRECTED_ROAD_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"road_flood_grounding.gpkg"
            )
        )

        directed_grounding_gdf.to_file(
            directed_gpkg_path,
            layer="road_flood_grounding",
            driver="GPKG",
        )

        directed_csv_path = (
            DIRECTED_ROAD_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"road_flood_grounding.csv"
            )
        )

        directed_grounding_gdf.drop(
            columns="geometry"
        ).to_csv(
            directed_csv_path,
            index=False,
        )

        # ----------------------------------------------------
        # Save physical-edge grounding
        # ----------------------------------------------------

        physical_edge_csv_path = (
            PHYSICAL_EDGE_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"physical_edge_grounding.csv"
            )
        )

        physical_edges_df.to_csv(
            physical_edge_csv_path,
            index=False,
        )

        physical_edge_json_path = (
            PHYSICAL_EDGE_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"physical_edge_grounding.json"
            )
        )

        with open(
            physical_edge_json_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                clean_dataframe_for_json(
                    physical_edges_df
                ),
                file,
                indent=2,
                ensure_ascii=False,
            )

        # ----------------------------------------------------
        # Save individual scene profile
        # ----------------------------------------------------

        scene_profile_csv_path = (
            SCENE_PROFILE_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"road_flood_profile.csv"
            )
        )

        pd.DataFrame(
            [
                scene_profile
            ]
        ).to_csv(
            scene_profile_csv_path,
            index=False,
        )

        scene_profile_json_path = (
            SCENE_PROFILE_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"road_flood_profile.json"
            )
        )

        with open(
            scene_profile_json_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                {
                    key: make_json_safe(
                        value
                    )
                    for key, value
                    in scene_profile.items()
                },
                file,
                indent=2,
                ensure_ascii=False,
            )

        # ----------------------------------------------------
        # Save individual scene token
        # ----------------------------------------------------

        scene_token_json_path = (
            SCENE_TOKEN_OUTPUT_DIR
            / (
                f"{scene_id}_"
                f"road_flood_token.json"
            )
        )

        with open(
            scene_token_json_path,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                scene_token_record,
                file,
                indent=2,
                ensure_ascii=False,
            )

        # ----------------------------------------------------
        # Add output paths to profile and manifest
        # ----------------------------------------------------

        scene_profile[
            "directed_grounding_gpkg_path"
        ] = str(
            directed_gpkg_path
        )

        scene_profile[
            "directed_grounding_csv_path"
        ] = str(
            directed_csv_path
        )

        scene_profile[
            "physical_edge_csv_path"
        ] = str(
            physical_edge_csv_path
        )

        scene_profile[
            "physical_edge_json_path"
        ] = str(
            physical_edge_json_path
        )

        scene_profile[
            "scene_profile_json_path"
        ] = str(
            scene_profile_json_path
        )

        scene_profile[
            "scene_token_json_path"
        ] = str(
            scene_token_json_path
        )

        processing_record[
            "directed_grounding_gpkg_path"
        ] = str(
            directed_gpkg_path
        )

        processing_record[
            "physical_edge_csv_path"
        ] = str(
            physical_edge_csv_path
        )

        all_scene_profiles.append(
            scene_profile
        )

        all_scene_tokens.append(
            scene_token_record
        )

        all_processing_records.append(
            processing_record
        )

        all_physical_edge_tables.append(
            physical_edges_df.copy()
        )

        print(
            f"  Directed records     : "
            f"{processing_record['directed_road_record_count']:,}"
        )

        print(
            f"  Physical edges       : "
            f"{processing_record['physical_edge_count']:,}"
        )

        print(
            f"  Flooded edges        : "
            f"{processing_record['event_flood_edge_count']:,}"
        )

        print(
            f"  Flooded critical     : "
            f"{processing_record['flooded_critical_edge_count']:,}"
        )

        print(
            f"  Flood burden         : "
            f"{scene_profile['scene_flood_burden']}"
        )

        print(
            f"  Critical disruption  : "
            f"{scene_profile['critical_network_disruption']}"
        )

        print(
            f"  Runtime              : "
            f"{processing_record['runtime_seconds']:.2f} sec"
        )

    except Exception as error:

        error_message = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )

        failure_record = {
            "scene_id": scene_id,
            "status": "failed",
            "directed_road_record_count": None,
            "physical_edge_count": None,
            "event_flood_edge_count": None,
            "flooded_critical_edge_count": None,
            "runtime_seconds": None,
            "error_message": error_message,
        }

        all_processing_records.append(
            failure_record
        )

        print(
            f"  FAILED: "
            f"{error_message}"
        )

        traceback.print_exc()


batch_runtime_seconds = round(
    time.time()
    - batch_start_time,
    2,
)


# ------------------------------------------------------------
# 7. Build master datasets
# ------------------------------------------------------------

master_scene_profiles_df = pd.DataFrame(
    all_scene_profiles
)

master_scene_tokens_df = pd.DataFrame(
    all_scene_tokens
)

processing_manifest_df = pd.DataFrame(
    all_processing_records
)

if all_physical_edge_tables:
    master_physical_edges_df = pd.concat(
        all_physical_edge_tables,
        ignore_index=True,
    )
else:
    master_physical_edges_df = pd.DataFrame()


# ------------------------------------------------------------
# 8. Save master scene profiles
# ------------------------------------------------------------

MASTER_SCENE_PROFILES_CSV = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_profiles.csv"
)

MASTER_SCENE_PROFILES_JSON = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_profiles.json"
)

master_scene_profiles_df.to_csv(
    MASTER_SCENE_PROFILES_CSV,
    index=False,
)

with open(
    MASTER_SCENE_PROFILES_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_scene_profiles_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 9. Save master scene tokens
# ------------------------------------------------------------

MASTER_SCENE_TOKENS_CSV = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_tokens.csv"
)

MASTER_SCENE_TOKENS_JSON = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_tokens.json"
)

master_scene_tokens_df.to_csv(
    MASTER_SCENE_TOKENS_CSV,
    index=False,
)

with open(
    MASTER_SCENE_TOKENS_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_scene_tokens_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 10. Save master physical-edge grounding
# ------------------------------------------------------------

MASTER_PHYSICAL_EDGES_CSV = (
    MASTER_OUTPUT_DIR
    / "master_physical_edge_flood_grounding.csv"
)

MASTER_PHYSICAL_EDGES_JSON = (
    MASTER_OUTPUT_DIR
    / "master_physical_edge_flood_grounding.json"
)

master_physical_edges_df.to_csv(
    MASTER_PHYSICAL_EDGES_CSV,
    index=False,
)

with open(
    MASTER_PHYSICAL_EDGES_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_physical_edges_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 11. Save processing manifest
# ------------------------------------------------------------

PROCESSING_MANIFEST_CSV = (
    MASTER_OUTPUT_DIR
    / "road_flood_processing_manifest.csv"
)

PROCESSING_MANIFEST_JSON = (
    MASTER_OUTPUT_DIR
    / "road_flood_processing_manifest.json"
)

processing_manifest_df.to_csv(
    PROCESSING_MANIFEST_CSV,
    index=False,
)

with open(
    PROCESSING_MANIFEST_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            processing_manifest_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 12. Build batch summary
# ------------------------------------------------------------

successful_scene_count = int(
    (
        processing_manifest_df[
            "status"
        ] == "success"
    ).sum()
)

failed_scene_count = int(
    (
        processing_manifest_df[
            "status"
        ] == "failed"
    ).sum()
)

batch_summary = {
    "requested_scene_count": int(
        total_scenes
    ),
    "successful_scene_count": (
        successful_scene_count
    ),
    "failed_scene_count": (
        failed_scene_count
    ),
    "master_scene_profile_count": int(
        len(
            master_scene_profiles_df
        )
    ),
    "master_scene_token_count": int(
        len(
            master_scene_tokens_df
        )
    ),
    "master_physical_edge_count": int(
        len(
            master_physical_edges_df
        )
    ),
    "total_event_flood_edge_count": int(
        master_scene_profiles_df[
            "event_flood_edge_count"
        ].sum()
    )
    if not master_scene_profiles_df.empty
    else 0,
    "total_flooded_critical_edge_count": int(
        master_scene_profiles_df[
            "flooded_critical_edge_count"
        ].sum()
    )
    if not master_scene_profiles_df.empty
    else 0,
    "batch_runtime_seconds": (
        batch_runtime_seconds
    ),
}

BATCH_SUMMARY_CSV = (
    MASTER_OUTPUT_DIR
    / "road_flood_batch_summary.csv"
)

BATCH_SUMMARY_JSON = (
    MASTER_OUTPUT_DIR
    / "road_flood_batch_summary.json"
)

pd.DataFrame(
    [
        batch_summary
    ]
).to_csv(
    BATCH_SUMMARY_CSV,
    index=False,
)

with open(
    BATCH_SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        batch_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 13. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BATCH ROAD–FLOOD GROUNDING COMPLETE")
print("=" * 80)

print(
    f"Requested scenes             : "
    f"{total_scenes:,}"
)

print(
    f"Successful scenes            : "
    f"{successful_scene_count:,}"
)

print(
    f"Failed scenes                : "
    f"{failed_scene_count:,}"
)

print(
    f"Master scene profiles        : "
    f"{len(master_scene_profiles_df):,}"
)

print(
    f"Master scene tokens          : "
    f"{len(master_scene_tokens_df):,}"
)

print(
    f"Master physical edges        : "
    f"{len(master_physical_edges_df):,}"
)

print(
    f"Total flood-exposed edges    : "
    f"{batch_summary['total_event_flood_edge_count']:,}"
)

print(
    f"Flooded critical edges       : "
    f"{batch_summary['total_flooded_critical_edge_count']:,}"
)

print(
    f"Total batch runtime          : "
    f"{batch_runtime_seconds:.2f} seconds"
)


# ------------------------------------------------------------
# 14. Alignment checks
# ------------------------------------------------------------

profile_scene_ids = set(
    master_scene_profiles_df[
        "scene_id"
    ].astype(str)
)

token_scene_ids = set(
    master_scene_tokens_df[
        "scene_id"
    ].astype(str)
)

successful_scene_ids = set(
    processing_manifest_df.loc[
        processing_manifest_df[
            "status"
        ] == "success",
        "scene_id",
    ].astype(str)
)

print("\nALIGNMENT VALIDATION")
print("-" * 80)

print(
    f"Profiles match tokens        : "
    f"{profile_scene_ids == token_scene_ids}"
)

print(
    f"Profiles match successes     : "
    f"{profile_scene_ids == successful_scene_ids}"
)

print(
    f"Duplicate scene profiles     : "
    f"{master_scene_profiles_df['scene_id'].duplicated().sum():,}"
)

print(
    f"Duplicate scene tokens       : "
    f"{master_scene_tokens_df['scene_id'].duplicated().sum():,}"
)

print(
    f"Missing scene tokens         : "
    f"{len(profile_scene_ids - token_scene_ids):,}"
)


# ------------------------------------------------------------
# 15. Output paths
# ------------------------------------------------------------

print("\nMASTER OUTPUT FILES")
print("-" * 80)

master_output_paths = [
    MASTER_SCENE_PROFILES_CSV,
    MASTER_SCENE_PROFILES_JSON,
    MASTER_SCENE_TOKENS_CSV,
    MASTER_SCENE_TOKENS_JSON,
    MASTER_PHYSICAL_EDGES_CSV,
    MASTER_PHYSICAL_EDGES_JSON,
    PROCESSING_MANIFEST_CSV,
    PROCESSING_MANIFEST_JSON,
    BATCH_SUMMARY_CSV,
    BATCH_SUMMARY_JSON,
]

for output_path in master_output_paths:
    print(output_path)


# ------------------------------------------------------------
# 16. Result previews
# ------------------------------------------------------------

print("\nSCENE PROFILE PREVIEW")
print("-" * 80)

profile_preview_columns = [
    "scene_id",
    "physical_edge_count",
    "edges_with_valid_coverage_count",
    "event_flood_edge_count",
    "event_flood_edge_share",
    "flooded_critical_edge_count",
    "flooded_critical_low_redundancy_count",
    "flooded_physical_bridge_count",
    "flooded_bridge_bottleneck_count",
    "flooded_major_road_bottleneck_count",
    "scene_flood_burden",
    "critical_network_disruption",
]

display(
    master_scene_profiles_df[
        profile_preview_columns
    ]
    .sort_values(
        [
            "event_flood_edge_share",
            "flooded_critical_edge_count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print("\nPROCESSING MANIFEST")
print("-" * 80)

display(
    processing_manifest_df
)


print("\nSCENE TOKEN PREVIEW")
print("-" * 80)

display(
    master_scene_tokens_df.head()
)

print("=" * 80)

BATCH ROAD–FLOOD GROUNDING FOR ALL SCENES

OUTPUT DIRECTORIES
--------------------------------------------------------------------------------
Batch root            : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding
Directed-road outputs : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/directed_road_grounding
Physical-edge outputs : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/physical_edge_grounding
Scene profiles        : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/scene_profiles
Scene tokens          : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/scene_tokens
Master outputs        : /home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master

BATCH INVENTORY
-------------------------------------------------------

  Directed records     : 2
  Physical edges       : 1
  Flooded edges        : 0
  Flooded critical     : 0
  Flood burden         : Minimal
  Critical disruption  : None
  Runtime              : 0.18 sec

[02/26] Processing India_1018327


  Directed records     : 54
  Physical edges       : 27
  Flooded edges        : 7
  Flooded critical     : 3
  Flood burden         : High
  Critical disruption  : High
  Runtime              : 0.21 sec

[03/26] Processing India_1050276


  Directed records     : 186
  Physical edges       : 100
  Flooded edges        : 17
  Flooded critical     : 5
  Flood burden         : Moderate
  Critical disruption  : High
  Runtime              : 0.41 sec

[04/26] Processing India_1068117


  Directed records     : 111
  Physical edges       : 60
  Flooded edges        : 14
  Flooded critical     : 2
  Flood burden         : High
  Critical disruption  : Moderate
  Runtime              : 0.31 sec

[05/26] Processing India_285297


...

,scene_id,physical_edge_count,edges_with_valid_coverage_count,event_flood_edge_count,event_flood_edge_share,flooded_critical_edge_count,flooded_critical_low_redundancy_count,flooded_physical_bridge_count,flooded_bridge_bottleneck_count,flooded_major_road_bottleneck_count,scene_flood_burden,critical_network_disruption
0,India_500266,67,53,32,0.6038,6,3,1,0,1,High,High
1,Somalia_970508,581,135,69,0.5111,24,2,0,0,0,High,Moderate
2,Nigeria_598959,27,12,6,0.5000,1,1,0,0,0,High,Moderate
3,India_956930,5,5,2,0.4000,2,2,2,2,0,High,High
4,Nigeria_1095404,7,5,2,0.4000,1,1,0,0,0,High,Moderate
5,India_943439,5,5,2,0.4000,0,0,0,0,0,High,None
6,Mekong_16233,26,15,5,0.3333,3,3,0,0,0,High,High
7,Somalia_322855,3,3,1,0.3333,1,1,0,0,0,High,High
8,India_383430,20,18,6,0.3333,0,0,3,3,0,High,High
9,India_1018327,27,24,7,0.2917,3,2,1,1,1,High,High



PROCESSING MANIFEST
--------------------------------------------------------------------------------


,scene_id,status,directed_road_record_count,physical_edge_count,event_flood_edge_count,flooded_critical_edge_count,runtime_seconds,error_message,directed_grounding_gpkg_path,physical_edge_csv_path
0,Ghana_141910,success,2,1,0,0,0.18,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
1,India_1018327,success,54,27,7,3,0.21,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
2,India_1050276,success,186,100,17,5,0.41,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
3,India_1068117,success,111,60,14,2,0.31,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
4,India_285297,success,68,42,8,2,0.23,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
5,India_383430,success,40,20,6,0,0.19,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
6,India_500266,success,134,67,32,6,0.32,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
7,India_773682,success,212,109,17,6,0.43,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
8,India_804466,success,173,94,11,1,0.31,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...
9,India_943439,success,10,5,2,0,0.14,None,/home/adjeiowusu1/myproject/ResilientVLM/data/...,/home/adjeiowusu1/myproject/ResilientVLM/data/...



SCENE TOKEN PREVIEW
--------------------------------------------------------------------------------


,scene_id,scene_flood_burden,critical_network_disruption,scene_road_flood_token
0,Ghana_141910,Minimal,None,<SceneFloodBurden:Minimal> <CriticalNetworkDis...
1,India_1018327,High,High,<SceneFloodBurden:High> <CriticalNetworkDisrup...
2,India_1050276,Moderate,High,<SceneFloodBurden:Moderate> <CriticalNetworkDi...
3,India_1068117,High,Moderate,<SceneFloodBurden:High> <CriticalNetworkDisrup...
4,India_285297,High,Moderate,<SceneFloodBurden:High> <CriticalNetworkDisrup...


In [8]:
# ============================================================
# CELL 8: FINAL VALIDATION AND COMBINED GROUNDING DATASET
# ============================================================

import json
import numpy as np
import pandas as pd

print("=" * 80)
print("FINAL ROAD–FLOOD GROUNDING VALIDATION")
print("=" * 80)


# ------------------------------------------------------------
# 1. Required scene-profile fields
# ------------------------------------------------------------

required_profile_columns = [
    "scene_id",
    "directed_road_record_count",
    "physical_edge_count",
    "edges_with_valid_coverage_count",
    "event_flood_edge_count",
    "event_flood_edge_share",
    "critical_edge_count",
    "flooded_critical_edge_count",
    "critical_low_redundancy_count",
    "flooded_critical_low_redundancy_count",
    "physical_bridge_count",
    "flooded_physical_bridge_count",
    "bridge_bottleneck_count",
    "flooded_bridge_bottleneck_count",
    "major_road_bottleneck_count",
    "flooded_major_road_bottleneck_count",
    "scene_flood_burden",
    "critical_network_disruption",
    "scene_road_flood_token",
]

missing_profile_columns = [
    column
    for column in required_profile_columns
    if column not in master_scene_profiles_df.columns
]

print("\nREQUIRED FIELD VALIDATION")
print("-" * 80)

print(
    f"Missing required profile fields: "
    f"{missing_profile_columns}"
)

if missing_profile_columns:
    raise ValueError(
        "Master scene profiles are missing required fields."
    )


# ------------------------------------------------------------
# 2. Structural validation
# ------------------------------------------------------------

expected_scene_ids = set(
    batch_inventory_df[
        "scene_id"
    ].astype(str)
)

profile_scene_ids = set(
    master_scene_profiles_df[
        "scene_id"
    ].astype(str)
)

token_scene_ids = set(
    master_scene_tokens_df[
        "scene_id"
    ].astype(str)
)

physical_edge_scene_ids = set(
    master_physical_edges_df[
        "scene_id"
    ].astype(str)
)

print("\nSCENE ALIGNMENT")
print("-" * 80)

print(
    f"Expected scenes              : "
    f"{len(expected_scene_ids):,}"
)

print(
    f"Profile scenes               : "
    f"{len(profile_scene_ids):,}"
)

print(
    f"Token scenes                 : "
    f"{len(token_scene_ids):,}"
)

print(
    f"Physical-edge scenes         : "
    f"{len(physical_edge_scene_ids):,}"
)

print(
    f"Expected = profiles          : "
    f"{expected_scene_ids == profile_scene_ids}"
)

print(
    f"Profiles = tokens            : "
    f"{profile_scene_ids == token_scene_ids}"
)

print(
    f"Profiles = physical edges    : "
    f"{profile_scene_ids == physical_edge_scene_ids}"
)

print(
    f"Duplicate profiles           : "
    f"{master_scene_profiles_df['scene_id'].duplicated().sum():,}"
)

print(
    f"Duplicate scene tokens       : "
    f"{master_scene_tokens_df['scene_id'].duplicated().sum():,}"
)


# ------------------------------------------------------------
# 3. Physical-edge count reconciliation
# ------------------------------------------------------------

physical_edge_counts_df = (
    master_physical_edges_df
    .groupby(
        "scene_id",
        as_index=False,
    )
    .agg(
        physical_edge_rows=(
            "physical_edge_id",
            "size",
        ),
        unique_physical_edge_ids=(
            "physical_edge_id",
            "nunique",
        ),
    )
)

count_validation_df = (
    master_scene_profiles_df[
        [
            "scene_id",
            "physical_edge_count",
        ]
    ]
    .merge(
        physical_edge_counts_df,
        on="scene_id",
        how="left",
    )
)

count_validation_df[
    "profile_matches_physical_table"
] = (
    count_validation_df[
        "physical_edge_count"
    ]
    == count_validation_df[
        "physical_edge_rows"
    ]
)

count_validation_df[
    "physical_ids_unique"
] = (
    count_validation_df[
        "physical_edge_rows"
    ]
    == count_validation_df[
        "unique_physical_edge_ids"
    ]
)

print("\nPHYSICAL-EDGE RECONCILIATION")
print("-" * 80)

print(
    f"Profiles matching physical tables: "
    f"{count_validation_df['profile_matches_physical_table'].sum():,}/"
    f"{len(count_validation_df):,}"
)

print(
    f"Scenes with unique edge IDs       : "
    f"{count_validation_df['physical_ids_unique'].sum():,}/"
    f"{len(count_validation_df):,}"
)

if not count_validation_df[
    "profile_matches_physical_table"
].all():

    print("\nCOUNT MISMATCHES")

    display(
        count_validation_df.loc[
            ~count_validation_df[
                "profile_matches_physical_table"
            ]
        ]
    )


# ------------------------------------------------------------
# 4. Numeric logic validation
# ------------------------------------------------------------

validation_df = (
    master_scene_profiles_df.copy()
)

validation_df[
    "flood_edges_within_valid_edges"
] = (
    validation_df[
        "event_flood_edge_count"
    ]
    <= validation_df[
        "edges_with_valid_coverage_count"
    ]
)

validation_df[
    "flooded_critical_within_critical"
] = (
    validation_df[
        "flooded_critical_edge_count"
    ]
    <= validation_df[
        "critical_edge_count"
    ]
)

validation_df[
    "flooded_low_redundancy_within_total"
] = (
    validation_df[
        "flooded_critical_low_redundancy_count"
    ]
    <= validation_df[
        "critical_low_redundancy_count"
    ]
)

validation_df[
    "flooded_bridges_within_total"
] = (
    validation_df[
        "flooded_physical_bridge_count"
    ]
    <= validation_df[
        "physical_bridge_count"
    ]
)

validation_df[
    "edge_share_valid"
] = (
    validation_df[
        "event_flood_edge_share"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
)

validation_df[
    "coverage_share_valid"
] = (
    validation_df[
        "raster_coverage_share"
    ]
    .between(
        0,
        1,
        inclusive="both",
    )
)

logic_columns = [
    "flood_edges_within_valid_edges",
    "flooded_critical_within_critical",
    "flooded_low_redundancy_within_total",
    "flooded_bridges_within_total",
    "edge_share_valid",
    "coverage_share_valid",
]

validation_df[
    "all_numeric_checks_pass"
] = validation_df[
    logic_columns
].all(axis=1)

print("\nNUMERIC LOGIC CHECKS")
print("-" * 80)

print(
    f"Scenes passing all checks: "
    f"{validation_df['all_numeric_checks_pass'].sum():,}/"
    f"{len(validation_df):,}"
)

if not validation_df[
    "all_numeric_checks_pass"
].all():

    print("\nSCENES FAILING NUMERIC CHECKS")

    display(
        validation_df.loc[
            ~validation_df[
                "all_numeric_checks_pass"
            ],
            [
                "scene_id",
                *logic_columns,
            ],
        ]
    )


# ------------------------------------------------------------
# 5. Coverage warnings
# ------------------------------------------------------------

LOW_ROAD_COUNT_THRESHOLD = 10
LOW_RASTER_COVERAGE_THRESHOLD = 0.50

master_scene_profiles_df[
    "limited_road_network_warning"
] = (
    master_scene_profiles_df[
        "physical_edge_count"
    ]
    < LOW_ROAD_COUNT_THRESHOLD
)

master_scene_profiles_df[
    "limited_raster_coverage_warning"
] = (
    master_scene_profiles_df[
        "raster_coverage_share"
    ]
    < LOW_RASTER_COVERAGE_THRESHOLD
)

master_scene_profiles_df[
    "grounding_quality_warning"
] = np.select(
        [
            (
                master_scene_profiles_df[
                    "limited_road_network_warning"
                ]
                &
                master_scene_profiles_df[
                    "limited_raster_coverage_warning"
                ]
            ),
            master_scene_profiles_df[
                "limited_road_network_warning"
            ],
            master_scene_profiles_df[
                "limited_raster_coverage_warning"
            ],
        ],
        [
            (
                "Limited road network and "
                "limited raster coverage"
            ),
            "Limited road network",
            "Limited raster coverage",
        ],
        default="None",
    )

print("\nGROUNDING QUALITY WARNINGS")
print("-" * 80)

warning_df = (
    master_scene_profiles_df.loc[
        master_scene_profiles_df[
            "grounding_quality_warning"
        ]
        != "None",
        [
            "scene_id",
            "physical_edge_count",
            "raster_coverage_share",
            "grounding_quality_warning",
        ],
    ]
    .sort_values(
        [
            "physical_edge_count",
            "raster_coverage_share",
        ]
    )
)

print(
    f"Scenes with warnings: "
    f"{len(warning_df):,}"
)

display(
    warning_df
)


# ------------------------------------------------------------
# 6. Rank scenes by transportation flood disruption
# ------------------------------------------------------------

ranking_df = (
    master_scene_profiles_df.copy()
)

ranking_df[
    "normalized_flooded_critical_count"
] = (
    ranking_df[
        "flooded_critical_edge_count"
    ]
    / max(
        ranking_df[
            "flooded_critical_edge_count"
        ].max(),
        1,
    )
)

ranking_df[
    "normalized_flooded_low_redundancy_count"
] = (
    ranking_df[
        "flooded_critical_low_redundancy_count"
    ]
    / max(
        ranking_df[
            "flooded_critical_low_redundancy_count"
        ].max(),
        1,
    )
)

ranking_df[
    "normalized_flooded_bottleneck_count"
] = (
    (
        ranking_df[
            "flooded_bridge_bottleneck_count"
        ]
        +
        ranking_df[
            "flooded_major_road_bottleneck_count"
        ]
    )
    / max(
        (
            ranking_df[
                "flooded_bridge_bottleneck_count"
            ]
            +
            ranking_df[
                "flooded_major_road_bottleneck_count"
            ]
        ).max(),
        1,
    )
)

ranking_df[
    "transportation_flood_disruption_score"
] = (
    0.35
    * ranking_df[
        "event_flood_edge_share"
    ]
    +
    0.30
    * ranking_df[
        "normalized_flooded_critical_count"
    ]
    +
    0.20
    * ranking_df[
        "normalized_flooded_low_redundancy_count"
    ]
    +
    0.15
    * ranking_df[
        "normalized_flooded_bottleneck_count"
    ]
).round(4)

ranking_df[
    "transportation_flood_disruption_rank"
] = (
    ranking_df[
        "transportation_flood_disruption_score"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)

ranking_df = ranking_df.sort_values(
    [
        "transportation_flood_disruption_rank",
        "scene_id",
    ]
).reset_index(drop=True)

print("\nTRANSPORTATION FLOOD-DISRUPTION RANKING")
print("-" * 80)

display(
    ranking_df[
        [
            "transportation_flood_disruption_rank",
            "scene_id",
            "physical_edge_count",
            "event_flood_edge_count",
            "event_flood_edge_share",
            "flooded_critical_edge_count",
            "flooded_critical_low_redundancy_count",
            "flooded_bridge_bottleneck_count",
            "flooded_major_road_bottleneck_count",
            "transportation_flood_disruption_score",
            "scene_flood_burden",
            "critical_network_disruption",
            "grounding_quality_warning",
        ]
    ].head(25)
)


# ------------------------------------------------------------
# 7. Merge Notebook 05 transportation tokens
# ------------------------------------------------------------

transport_token_column_candidates = [
    "transportation_token",
    "scene_transportation_token",
    "transportation_knowledge_token",
    "scene_token",
]

transport_token_column = next(
    (
        column
        for column in transport_token_column_candidates
        if column
        in master_transportation_tokens_df.columns
    ),
    None,
)

if transport_token_column is None:
    non_scene_columns = [
        column
        for column
        in master_transportation_tokens_df.columns
        if column != "scene_id"
    ]

    if len(non_scene_columns) == 1:
        transport_token_column = (
            non_scene_columns[0]
        )
    else:
        raise ValueError(
            "Could not identify the Notebook 05 "
            "transportation-token column. Available columns: "
            f"{master_transportation_tokens_df.columns.tolist()}"
        )

print("\nTRANSPORTATION TOKEN COLUMN")
print("-" * 80)

print(
    f"Selected column: "
    f"{transport_token_column}"
)

combined_grounding_df = (
    ranking_df
    .merge(
        master_transportation_tokens_df[
            [
                "scene_id",
                transport_token_column,
            ]
        ].rename(
            columns={
                transport_token_column:
                "transportation_knowledge_token"
            }
        ),
        on="scene_id",
        how="left",
        validate="one_to_one",
    )
)

combined_grounding_df[
    "combined_transportation_flood_token"
] = (
    combined_grounding_df[
        "transportation_knowledge_token"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    +
    " "
    +
    combined_grounding_df[
        "scene_road_flood_token"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

combined_grounding_df[
    "combined_transportation_flood_token"
] = (
    combined_grounding_df[
        "combined_transportation_flood_token"
    ]
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

missing_transport_tokens = int(
    combined_grounding_df[
        "transportation_knowledge_token"
    ].isna().sum()
)

missing_flood_tokens = int(
    combined_grounding_df[
        "scene_road_flood_token"
    ].isna().sum()
)

print("\nCOMBINED TOKEN VALIDATION")
print("-" * 80)

print(
    f"Combined records             : "
    f"{len(combined_grounding_df):,}"
)

print(
    f"Missing transportation token: "
    f"{missing_transport_tokens:,}"
)

print(
    f"Missing flood token         : "
    f"{missing_flood_tokens:,}"
)

print(
    f"Duplicate combined scenes   : "
    f"{combined_grounding_df['scene_id'].duplicated().sum():,}"
)


# ------------------------------------------------------------
# 8. Create compact downstream dataset
# ------------------------------------------------------------

downstream_columns = [
    "scene_id",
    "physical_edge_count",
    "edges_with_valid_coverage_count",
    "raster_coverage_share",
    "event_flood_edge_count",
    "event_flood_edge_share",
    "low_flood_exposure_edge_count",
    "moderate_flood_exposure_edge_count",
    "high_flood_exposure_edge_count",
    "critical_edge_count",
    "flooded_critical_edge_count",
    "flooded_critical_edge_share",
    "critical_low_redundancy_count",
    "flooded_critical_low_redundancy_count",
    "flooded_critical_low_redundancy_share",
    "physical_bridge_count",
    "flooded_physical_bridge_count",
    "bridge_bottleneck_count",
    "flooded_bridge_bottleneck_count",
    "major_road_bottleneck_count",
    "flooded_major_road_bottleneck_count",
    "scene_flood_burden",
    "critical_network_disruption",
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
    "grounding_quality_warning",
    "transportation_knowledge_token",
    "scene_road_flood_token",
    "combined_transportation_flood_token",
]

combined_grounding_dataset_df = (
    combined_grounding_df[
        downstream_columns
    ]
    .copy()
    .sort_values(
        "scene_id"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. Save final Notebook 06 outputs
# ------------------------------------------------------------

FINAL_COMBINED_GROUNDING_CSV = (
    MASTER_OUTPUT_DIR
    / "master_combined_transportation_flood_grounding.csv"
)

FINAL_COMBINED_GROUNDING_JSON = (
    MASTER_OUTPUT_DIR
    / "master_combined_transportation_flood_grounding.json"
)

FINAL_DISRUPTION_RANKING_CSV = (
    MASTER_OUTPUT_DIR
    / "transportation_flood_disruption_ranking.csv"
)

FINAL_DISRUPTION_RANKING_JSON = (
    MASTER_OUTPUT_DIR
    / "transportation_flood_disruption_ranking.json"
)

FINAL_VALIDATION_CSV = (
    MASTER_OUTPUT_DIR
    / "road_flood_final_validation.csv"
)

FINAL_NOTEBOOK_SUMMARY_JSON = (
    MASTER_OUTPUT_DIR
    / "notebook_06_completion_summary.json"
)


combined_grounding_dataset_df.to_csv(
    FINAL_COMBINED_GROUNDING_CSV,
    index=False,
)

with open(
    FINAL_COMBINED_GROUNDING_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            combined_grounding_dataset_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


ranking_df.to_csv(
    FINAL_DISRUPTION_RANKING_CSV,
    index=False,
)

with open(
    FINAL_DISRUPTION_RANKING_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            ranking_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


final_validation_df = (
    validation_df[
        [
            "scene_id",
            *logic_columns,
            "all_numeric_checks_pass",
        ]
    ]
    .merge(
        count_validation_df[
            [
                "scene_id",
                "profile_matches_physical_table",
                "physical_ids_unique",
            ]
        ],
        on="scene_id",
        how="left",
    )
)

final_validation_df.to_csv(
    FINAL_VALIDATION_CSV,
    index=False,
)


completion_summary = {
    "notebook": "06_road_flood_grounding",
    "status": "complete",
    "scene_count": int(
        len(
            combined_grounding_dataset_df
        )
    ),
    "physical_edge_record_count": int(
        len(
            master_physical_edges_df
        )
    ),
    "all_expected_scenes_present": bool(
        expected_scene_ids
        == profile_scene_ids
    ),
    "all_scene_tokens_present": bool(
        missing_flood_tokens == 0
    ),
    "all_transportation_tokens_present": bool(
        missing_transport_tokens == 0
    ),
    "all_numeric_checks_pass": bool(
        validation_df[
            "all_numeric_checks_pass"
        ].all()
    ),
    "all_physical_edge_counts_match": bool(
        count_validation_df[
            "profile_matches_physical_table"
        ].all()
    ),
    "all_physical_edge_ids_unique": bool(
        count_validation_df[
            "physical_ids_unique"
        ].all()
    ),
    "scenes_with_quality_warnings": int(
        len(
            warning_df
        )
    ),
    "total_flood_exposed_physical_edges": int(
        master_scene_profiles_df[
            "event_flood_edge_count"
        ].sum()
    ),
    "total_flooded_critical_edges": int(
        master_scene_profiles_df[
            "flooded_critical_edge_count"
        ].sum()
    ),
    "outputs": {
        "combined_grounding_csv": str(
            FINAL_COMBINED_GROUNDING_CSV
        ),
        "combined_grounding_json": str(
            FINAL_COMBINED_GROUNDING_JSON
        ),
        "disruption_ranking_csv": str(
            FINAL_DISRUPTION_RANKING_CSV
        ),
        "disruption_ranking_json": str(
            FINAL_DISRUPTION_RANKING_JSON
        ),
        "validation_csv": str(
            FINAL_VALIDATION_CSV
        ),
    },
}

with open(
    FINAL_NOTEBOOK_SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 10. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK 06 COMPLETION REPORT")
print("=" * 80)

print(
    f"Combined grounding scenes       : "
    f"{len(combined_grounding_dataset_df):,}"
)

print(
    f"Physical-edge grounding records : "
    f"{len(master_physical_edges_df):,}"
)

print(
    f"All expected scenes present     : "
    f"{completion_summary['all_expected_scenes_present']}"
)

print(
    f"All transportation tokens found: "
    f"{completion_summary['all_transportation_tokens_present']}"
)

print(
    f"All flood tokens found          : "
    f"{completion_summary['all_scene_tokens_present']}"
)

print(
    f"All numeric checks passed       : "
    f"{completion_summary['all_numeric_checks_pass']}"
)

print(
    f"All edge counts reconciled      : "
    f"{completion_summary['all_physical_edge_counts_match']}"
)

print(
    f"All physical-edge IDs unique    : "
    f"{completion_summary['all_physical_edge_ids_unique']}"
)

print(
    f"Scenes with quality warnings    : "
    f"{completion_summary['scenes_with_quality_warnings']:,}"
)

print(
    f"Flood-exposed physical edges    : "
    f"{completion_summary['total_flood_exposed_physical_edges']:,}"
)

print(
    f"Flooded critical physical edges : "
    f"{completion_summary['total_flooded_critical_edges']:,}"
)

print("\nFINAL OUTPUT FILES")
print("-" * 80)

for path in [
    FINAL_COMBINED_GROUNDING_CSV,
    FINAL_COMBINED_GROUNDING_JSON,
    FINAL_DISRUPTION_RANKING_CSV,
    FINAL_DISRUPTION_RANKING_JSON,
    FINAL_VALIDATION_CSV,
    FINAL_NOTEBOOK_SUMMARY_JSON,
]:
    print(path)


print("\nCOMBINED DATASET PREVIEW")
print("-" * 80)

display(
    combined_grounding_dataset_df[
        [
            "scene_id",
            "event_flood_edge_count",
            "event_flood_edge_share",
            "flooded_critical_edge_count",
            "transportation_flood_disruption_score",
            "transportation_flood_disruption_rank",
            "scene_flood_burden",
            "critical_network_disruption",
            "grounding_quality_warning",
        ]
    ]
    .sort_values(
        "transportation_flood_disruption_rank"
    )
    .head(15)
)


print("\nCOMBINED TOKEN EXAMPLE")
print("-" * 80)

example_combined_row = (
    combined_grounding_dataset_df
    .sort_values(
        "transportation_flood_disruption_rank"
    )
    .iloc[0]
)

print(
    f"Scene: "
    f"{example_combined_row['scene_id']}"
)

print(
    example_combined_row[
        "combined_transportation_flood_token"
    ]
)

print("=" * 80)

FINAL ROAD–FLOOD GROUNDING VALIDATION

REQUIRED FIELD VALIDATION
--------------------------------------------------------------------------------
Missing required profile fields: []

SCENE ALIGNMENT
--------------------------------------------------------------------------------
Expected scenes              : 26
Profile scenes               : 26
Token scenes                 : 26
Physical-edge scenes         : 26
Expected = profiles          : True
Profiles = tokens            : True
Profiles = physical edges    : True
Duplicate profiles           : 0
Duplicate scene tokens       : 0

PHYSICAL-EDGE RECONCILIATION
--------------------------------------------------------------------------------
Profiles matching physical tables: 26/26
Scenes with unique edge IDs       : 26/26

NUMERIC LOGIC CHECKS
--------------------------------------------------------------------------------
Scenes passing all checks: 26/26

GROUNDING QUALITY WARNINGS
----------------------------------------------------

,scene_id,physical_edge_count,raster_coverage_share,grounding_quality_warning
0,Ghana_141910,1,1.0000,Limited road network
14,Somalia_322855,3,1.0000,Limited road network
9,India_943439,5,1.0000,Limited road network
10,India_956930,5,1.0000,Limited road network
12,Nigeria_1095404,7,0.7143,Limited road network
13,Nigeria_598959,27,0.4815,Limited raster coverage
15,Somalia_970508,581,0.4182,Limited raster coverage



TRANSPORTATION FLOOD-DISRUPTION RANKING
--------------------------------------------------------------------------------


,transportation_flood_disruption_rank,scene_id,physical_edge_count,event_flood_edge_count,event_flood_edge_share,flooded_critical_edge_count,flooded_critical_low_redundancy_count,flooded_bridge_bottleneck_count,flooded_major_road_bottleneck_count,transportation_flood_disruption_score,scene_flood_burden,critical_network_disruption,grounding_quality_warning
0,1,Spain_7370579,1309,175,0.1649,63,10,2,3,0.7077,Moderate,High,None
1,2,Spain_1167260,1063,129,0.1352,20,6,2,1,0.3526,Moderate,High,None
2,3,Spain_8565131,542,53,0.1121,20,7,2,0,0.3345,Moderate,Moderate,None
3,4,Somalia_970508,581,69,0.5111,24,2,0,0,0.3332,High,Moderate,Limited raster coverage
4,5,India_500266,67,32,0.6038,6,3,0,1,0.3299,High,High,None
5,6,India_1050276,100,17,0.2099,5,4,4,1,0.3273,Moderate,High,None
6,7,India_956930,5,2,0.4000,2,2,2,0,0.2495,High,High,Limited road network
7,8,India_1018327,27,7,0.2917,3,2,1,1,0.2164,High,High,None
8,9,India_383430,20,6,0.3333,0,0,3,0,0.2067,High,High,None
9,10,Nigeria_598959,27,6,0.5000,1,1,0,0,0.1998,High,Moderate,Limited raster coverage



TRANSPORTATION TOKEN COLUMN
--------------------------------------------------------------------------------
Selected column: scene_transportation_token

COMBINED TOKEN VALIDATION
--------------------------------------------------------------------------------
Combined records             : 26
Missing transportation token: 0
Missing flood token         : 0
Duplicate combined scenes   : 0

NOTEBOOK 06 COMPLETION REPORT
Combined grounding scenes       : 26
Physical-edge grounding records : 6,093
All expected scenes present     : True
All transportation tokens found: True
All flood tokens found          : True
All numeric checks passed       : True
All edge counts reconciled      : True
All physical-edge IDs unique    : True
Scenes with quality warnings    : 7
Flood-exposed physical edges    : 645
Flooded critical physical edges : 193

FINAL OUTPUT FILES
--------------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processe

,scene_id,event_flood_edge_count,event_flood_edge_share,flooded_critical_edge_count,transportation_flood_disruption_score,transportation_flood_disruption_rank,scene_flood_burden,critical_network_disruption,grounding_quality_warning
17,Spain_7370579,175,0.1649,63,0.7077,1,Moderate,High,None
16,Spain_1167260,129,0.1352,20,0.3526,2,Moderate,High,None
18,Spain_8565131,53,0.1121,20,0.3345,3,Moderate,Moderate,None
15,Somalia_970508,69,0.5111,24,0.3332,4,High,Moderate,Limited raster coverage
6,India_500266,32,0.6038,6,0.3299,5,High,High,None
2,India_1050276,17,0.2099,5,0.3273,6,Moderate,High,None
10,India_956930,2,0.4000,2,0.2495,7,High,High,Limited road network
1,India_1018327,7,0.2917,3,0.2164,8,High,High,None
5,India_383430,6,0.3333,0,0.2067,9,High,High,None
13,Nigeria_598959,6,0.5000,1,0.1998,10,High,Moderate,Limited raster coverage



COMBINED TOKEN EXAMPLE
--------------------------------------------------------------------------------
Scene: Spain_7370579
<NetworkConnectivity:High> <SingleAccessExposure:Low> <AlternateRouteAvailability:High> <CriticalCorridorBurden:Moderate> <TopologicalVulnerabilityBurden:Moderate> <BridgeBottlenecks:Present> <MajorRoadBottlenecks:Present> <PhysicalBridges:Present> <NetworkFragmentation:Present> <SceneFloodBurden:Moderate> <CriticalNetworkDisruption:High> <FloodedCriticalEdges:Yes> <FloodedLowRedundancyEdges:Yes> <FloodedPhysicalBridges:Yes> <FloodedBridgeBottlenecks:Yes> <FloodedMajorRoadBottlenecks:Yes> <RoadRasterCoverage:0.8105> <FloodedRoadShare:0.1649>


In [9]:
# ============================================================
# CELL 9: FINALIZE PROVENANCE AND RANKING METADATA
# ============================================================

from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd


print("=" * 80)
print("FINALIZING NOTEBOOK 06 PROVENANCE AND RANKING METADATA")
print("=" * 80)


# ------------------------------------------------------------
# 1. Methodology and provenance constants
# ------------------------------------------------------------

GROUNDING_VERSION = "v1.1"

FLOOD_DEFINITION = (
    "Event flood proxy = "
    "(LabelHand == 1) AND (JRCWaterHand == 0)"
)

SURFACE_WATER_DEFINITION = (
    "Surface water = LabelHand == 1"
)

PERMANENT_WATER_DEFINITION = (
    "Permanent water = JRCWaterHand == 1"
)

INVALID_PIXEL_DEFINITION = (
    "Invalid pixel = LabelHand == -1"
)

LABEL_SOURCE_NAME = "Sen1Floods11 LabelHand"

PERMANENT_WATER_SOURCE_NAME = (
    "Sen1Floods11 JRCWaterHand"
)

ROAD_SOURCE_NAME = (
    "OpenStreetMap-derived transportation network"
)

TRANSPORTATION_KNOWLEDGE_SOURCE = (
    "Notebook 05 transportation knowledge outputs"
)

ROAD_BUFFER_METHOD = (
    "Road centerlines projected to scene-specific UTM CRS "
    "and buffered by 10 meters before raster intersection"
)

PHYSICAL_EDGE_DEFINITION = (
    "Directed u-to-v and v-to-u road records collapsed "
    "to one physical edge using sorted endpoint IDs and key"
)

EXPOSURE_CLASSIFICATION_METHOD = (
    "No detected: event-flood fraction < 0.01; "
    "Low: 0.01 to < 0.10; "
    "Moderate: 0.10 to < 0.30; "
    "High: >= 0.30"
)

SCENE_FLOOD_BURDEN_METHOD = (
    "Minimal: flooded-road share < 0.01; "
    "Low: 0.01 to < 0.10; "
    "Moderate: 0.10 to < 0.25; "
    "High: >= 0.25"
)

DISRUPTION_SCORE_METHOD = (
    "0.35 * flooded-road share + "
    "0.30 * normalized flooded-critical-edge count + "
    "0.20 * normalized flooded-low-redundancy count + "
    "0.15 * normalized flooded-bottleneck count"
)

GROUNDING_GENERATED_UTC = (
    datetime.now(timezone.utc)
    .replace(microsecond=0)
    .isoformat()
)


# ------------------------------------------------------------
# 2. Prepare score and rank table
# ------------------------------------------------------------

required_ranking_columns = [
    "scene_id",
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
]

missing_ranking_columns = [
    column
    for column in required_ranking_columns
    if column not in ranking_df.columns
]

if missing_ranking_columns:
    raise ValueError(
        "Ranking table is missing required columns: "
        f"{missing_ranking_columns}"
    )

ranking_metadata_df = (
    ranking_df[
        required_ranking_columns
    ]
    .copy()
)

ranking_metadata_df[
    "scene_id"
] = ranking_metadata_df[
    "scene_id"
].astype(str)

if ranking_metadata_df[
    "scene_id"
].duplicated().any():
    raise ValueError(
        "Duplicate scene IDs found in ranking metadata."
    )


# ------------------------------------------------------------
# 3. Remove any existing ranking columns before merging
# ------------------------------------------------------------

profile_columns_to_replace = [
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
]

master_scene_profiles_updated_df = (
    master_scene_profiles_df
    .drop(
        columns=[
            column
            for column in profile_columns_to_replace
            if column in master_scene_profiles_df.columns
        ],
        errors="ignore",
    )
    .copy()
)

master_scene_profiles_updated_df[
    "scene_id"
] = master_scene_profiles_updated_df[
    "scene_id"
].astype(str)

master_scene_profiles_updated_df = (
    master_scene_profiles_updated_df
    .merge(
        ranking_metadata_df,
        on="scene_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 4. Add provenance metadata to scene profiles
# ------------------------------------------------------------

provenance_fields = {
    "grounding_version": GROUNDING_VERSION,
    "grounding_generated_utc": GROUNDING_GENERATED_UTC,

    "road_buffer_meters": float(
        ROAD_BUFFER_METERS
    ),

    "flood_definition": FLOOD_DEFINITION,
    "surface_water_definition": (
        SURFACE_WATER_DEFINITION
    ),
    "permanent_water_definition": (
        PERMANENT_WATER_DEFINITION
    ),
    "invalid_pixel_definition": (
        INVALID_PIXEL_DEFINITION
    ),

    "flood_label_source": LABEL_SOURCE_NAME,
    "permanent_water_source": (
        PERMANENT_WATER_SOURCE_NAME
    ),
    "road_network_source": ROAD_SOURCE_NAME,
    "transportation_knowledge_source": (
        TRANSPORTATION_KNOWLEDGE_SOURCE
    ),

    "road_buffer_method": ROAD_BUFFER_METHOD,
    "physical_edge_definition": (
        PHYSICAL_EDGE_DEFINITION
    ),
    "road_flood_exposure_method": (
        EXPOSURE_CLASSIFICATION_METHOD
    ),
    "scene_flood_burden_method": (
        SCENE_FLOOD_BURDEN_METHOD
    ),
    "disruption_score_method": (
        DISRUPTION_SCORE_METHOD
    ),

    "flood_touch_threshold": float(
        FLOOD_TOUCH_THRESHOLD
    ),
    "moderate_flood_threshold": float(
        MODERATE_FLOOD_THRESHOLD
    ),
    "high_flood_threshold": float(
        HIGH_FLOOD_THRESHOLD
    ),
}

for column, value in provenance_fields.items():
    master_scene_profiles_updated_df[
        column
    ] = value


# ------------------------------------------------------------
# 5. Add a reliability category
# ------------------------------------------------------------

def classify_grounding_reliability(row):
    """
    Convert quality warnings into a compact reliability label.
    """

    limited_roads = bool(
        row.get(
            "limited_road_network_warning",
            False,
        )
    )

    limited_coverage = bool(
        row.get(
            "limited_raster_coverage_warning",
            False,
        )
    )

    if limited_roads and limited_coverage:
        return "Low"

    if limited_roads or limited_coverage:
        return "Moderate"

    return "High"


master_scene_profiles_updated_df[
    "grounding_reliability"
] = master_scene_profiles_updated_df.apply(
    classify_grounding_reliability,
    axis=1,
)


# ------------------------------------------------------------
# 6. Build a human-readable interpretation note
# ------------------------------------------------------------

def build_scene_interpretation_note(row):
    """
    Create a concise interpretation of each scene's grounding.
    """

    scene_id = str(
        row["scene_id"]
    )

    flood_burden = str(
        row["scene_flood_burden"]
    ).lower()

    disruption = str(
        row["critical_network_disruption"]
    ).lower()

    flooded_edges = int(
        row["event_flood_edge_count"]
    )

    flooded_critical = int(
        row["flooded_critical_edge_count"]
    )

    reliability = str(
        row["grounding_reliability"]
    ).lower()

    note = (
        f"{scene_id} has a {flood_burden} road-flood burden "
        f"and {disruption} critical-network disruption, with "
        f"{flooded_edges} flood-exposed physical edges and "
        f"{flooded_critical} flooded critical edges. "
        f"Grounding reliability is {reliability}."
    )

    return note


master_scene_profiles_updated_df[
    "scene_interpretation_note"
] = (
    master_scene_profiles_updated_df.apply(
        build_scene_interpretation_note,
        axis=1,
    )
)


# ------------------------------------------------------------
# 7. Update combined grounding dataset
# ------------------------------------------------------------

combined_columns_to_replace = [
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
    "grounding_version",
    "grounding_generated_utc",
    "flood_definition",
    "surface_water_definition",
    "permanent_water_definition",
    "flood_label_source",
    "permanent_water_source",
    "road_network_source",
    "road_buffer_meters",
    "physical_edge_definition",
    "disruption_score_method",
    "grounding_reliability",
    "scene_interpretation_note",
]

combined_grounding_updated_df = (
    combined_grounding_dataset_df
    .drop(
        columns=[
            column
            for column
            in combined_columns_to_replace
            if column
            in combined_grounding_dataset_df.columns
        ],
        errors="ignore",
    )
    .copy()
)

combined_grounding_updated_df[
    "scene_id"
] = combined_grounding_updated_df[
    "scene_id"
].astype(str)

metadata_columns_for_combined = [
    "scene_id",
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
    "grounding_version",
    "grounding_generated_utc",
    "flood_definition",
    "surface_water_definition",
    "permanent_water_definition",
    "flood_label_source",
    "permanent_water_source",
    "road_network_source",
    "road_buffer_meters",
    "physical_edge_definition",
    "disruption_score_method",
    "grounding_reliability",
    "scene_interpretation_note",
]

combined_grounding_updated_df = (
    combined_grounding_updated_df
    .merge(
        master_scene_profiles_updated_df[
            metadata_columns_for_combined
        ],
        on="scene_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values("scene_id")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Update master scene token table
# ------------------------------------------------------------

master_scene_tokens_updated_df = (
    master_scene_tokens_df.copy()
)

master_scene_tokens_updated_df[
    "scene_id"
] = master_scene_tokens_updated_df[
    "scene_id"
].astype(str)

token_metadata_df = (
    master_scene_profiles_updated_df[
        [
            "scene_id",
            "transportation_flood_disruption_score",
            "transportation_flood_disruption_rank",
            "grounding_reliability",
            "grounding_version",
        ]
    ]
    .copy()
)

token_columns_to_replace = [
    "transportation_flood_disruption_score",
    "transportation_flood_disruption_rank",
    "grounding_reliability",
    "grounding_version",
]

master_scene_tokens_updated_df = (
    master_scene_tokens_updated_df
    .drop(
        columns=[
            column
            for column in token_columns_to_replace
            if column
            in master_scene_tokens_updated_df.columns
        ],
        errors="ignore",
    )
    .merge(
        token_metadata_df,
        on="scene_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 9. Add provenance to the master physical-edge dataset
# ------------------------------------------------------------

master_physical_edges_updated_df = (
    master_physical_edges_df.copy()
)

master_physical_edges_updated_df[
    "grounding_version"
] = GROUNDING_VERSION

master_physical_edges_updated_df[
    "road_buffer_meters"
] = float(
    ROAD_BUFFER_METERS
)

master_physical_edges_updated_df[
    "flood_definition"
] = FLOOD_DEFINITION

master_physical_edges_updated_df[
    "flood_label_source"
] = LABEL_SOURCE_NAME

master_physical_edges_updated_df[
    "permanent_water_source"
] = PERMANENT_WATER_SOURCE_NAME


# ------------------------------------------------------------
# 10. Validation
# ------------------------------------------------------------

expected_scene_count = len(
    master_scene_profiles_df
)

validation_results = {
    "profile_count_preserved": (
        len(
            master_scene_profiles_updated_df
        )
        == expected_scene_count
    ),

    "combined_count_preserved": (
        len(
            combined_grounding_updated_df
        )
        == expected_scene_count
    ),

    "token_count_preserved": (
        len(
            master_scene_tokens_updated_df
        )
        == expected_scene_count
    ),

    "no_duplicate_profile_scenes": (
        master_scene_profiles_updated_df[
            "scene_id"
        ].duplicated().sum()
        == 0
    ),

    "no_duplicate_combined_scenes": (
        combined_grounding_updated_df[
            "scene_id"
        ].duplicated().sum()
        == 0
    ),

    "all_scores_present": (
        master_scene_profiles_updated_df[
            "transportation_flood_disruption_score"
        ].notna().all()
    ),

    "all_ranks_present": (
        master_scene_profiles_updated_df[
            "transportation_flood_disruption_rank"
        ].notna().all()
    ),

    "all_versions_present": (
        master_scene_profiles_updated_df[
            "grounding_version"
        ].notna().all()
    ),

    "all_reliability_labels_present": (
        master_scene_profiles_updated_df[
            "grounding_reliability"
        ].notna().all()
    ),
}

print("\nVALIDATION RESULTS")
print("-" * 80)

for check_name, check_result in (
    validation_results.items()
):
    print(
        f"{check_name:<40}: "
        f"{check_result}"
    )

if not all(
    validation_results.values()
):
    failed_checks = [
        check_name
        for check_name, check_result
        in validation_results.items()
        if not check_result
    ]

    raise ValueError(
        "Final metadata validation failed: "
        f"{failed_checks}"
    )


# ------------------------------------------------------------
# 11. Save updated canonical files
# ------------------------------------------------------------

UPDATED_MASTER_SCENE_PROFILES_CSV = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_profiles.csv"
)

UPDATED_MASTER_SCENE_PROFILES_JSON = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_profiles.json"
)

UPDATED_MASTER_SCENE_TOKENS_CSV = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_tokens.csv"
)

UPDATED_MASTER_SCENE_TOKENS_JSON = (
    MASTER_OUTPUT_DIR
    / "master_road_flood_scene_tokens.json"
)

UPDATED_MASTER_PHYSICAL_EDGES_CSV = (
    MASTER_OUTPUT_DIR
    / "master_physical_edge_flood_grounding.csv"
)

UPDATED_MASTER_PHYSICAL_EDGES_JSON = (
    MASTER_OUTPUT_DIR
    / "master_physical_edge_flood_grounding.json"
)

UPDATED_COMBINED_GROUNDING_CSV = (
    MASTER_OUTPUT_DIR
    / "master_combined_transportation_flood_grounding.csv"
)

UPDATED_COMBINED_GROUNDING_JSON = (
    MASTER_OUTPUT_DIR
    / "master_combined_transportation_flood_grounding.json"
)

PROVENANCE_CSV = (
    MASTER_OUTPUT_DIR
    / "road_flood_grounding_provenance.csv"
)

PROVENANCE_JSON = (
    MASTER_OUTPUT_DIR
    / "road_flood_grounding_provenance.json"
)


master_scene_profiles_updated_df.to_csv(
    UPDATED_MASTER_SCENE_PROFILES_CSV,
    index=False,
)

with open(
    UPDATED_MASTER_SCENE_PROFILES_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_scene_profiles_updated_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


master_scene_tokens_updated_df.to_csv(
    UPDATED_MASTER_SCENE_TOKENS_CSV,
    index=False,
)

with open(
    UPDATED_MASTER_SCENE_TOKENS_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_scene_tokens_updated_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


master_physical_edges_updated_df.to_csv(
    UPDATED_MASTER_PHYSICAL_EDGES_CSV,
    index=False,
)

with open(
    UPDATED_MASTER_PHYSICAL_EDGES_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            master_physical_edges_updated_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


combined_grounding_updated_df.to_csv(
    UPDATED_COMBINED_GROUNDING_CSV,
    index=False,
)

with open(
    UPDATED_COMBINED_GROUNDING_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_dataframe_for_json(
            combined_grounding_updated_df
        ),
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 12. Save standalone provenance record
# ------------------------------------------------------------

provenance_record = {
    "notebook": "06_road_flood_grounding",
    "grounding_version": GROUNDING_VERSION,
    "grounding_generated_utc": (
        GROUNDING_GENERATED_UTC
    ),
    "scene_count": int(
        len(
            master_scene_profiles_updated_df
        )
    ),
    "physical_edge_record_count": int(
        len(
            master_physical_edges_updated_df
        )
    ),
    "road_buffer_meters": float(
        ROAD_BUFFER_METERS
    ),
    "flood_definition": FLOOD_DEFINITION,
    "surface_water_definition": (
        SURFACE_WATER_DEFINITION
    ),
    "permanent_water_definition": (
        PERMANENT_WATER_DEFINITION
    ),
    "invalid_pixel_definition": (
        INVALID_PIXEL_DEFINITION
    ),
    "flood_label_source": LABEL_SOURCE_NAME,
    "permanent_water_source": (
        PERMANENT_WATER_SOURCE_NAME
    ),
    "road_network_source": ROAD_SOURCE_NAME,
    "transportation_knowledge_source": (
        TRANSPORTATION_KNOWLEDGE_SOURCE
    ),
    "road_buffer_method": ROAD_BUFFER_METHOD,
    "physical_edge_definition": (
        PHYSICAL_EDGE_DEFINITION
    ),
    "road_flood_exposure_method": (
        EXPOSURE_CLASSIFICATION_METHOD
    ),
    "scene_flood_burden_method": (
        SCENE_FLOOD_BURDEN_METHOD
    ),
    "disruption_score_method": (
        DISRUPTION_SCORE_METHOD
    ),
    "quality_warning_rules": {
        "limited_road_network": (
            "physical_edge_count < 10"
        ),
        "limited_raster_coverage": (
            "raster_coverage_share < 0.50"
        ),
    },
}

pd.DataFrame(
    [
        {
            key: (
                json.dumps(value)
                if isinstance(
                    value,
                    dict,
                )
                else value
            )
            for key, value
            in provenance_record.items()
        }
    ]
).to_csv(
    PROVENANCE_CSV,
    index=False,
)

with open(
    PROVENANCE_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        provenance_record,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 13. Update completion summary
# ------------------------------------------------------------

UPDATED_COMPLETION_SUMMARY_JSON = (
    MASTER_OUTPUT_DIR
    / "notebook_06_completion_summary.json"
)

if UPDATED_COMPLETION_SUMMARY_JSON.exists():

    with open(
        UPDATED_COMPLETION_SUMMARY_JSON,
        "r",
        encoding="utf-8",
    ) as file:
        updated_completion_summary = (
            json.load(file)
        )

else:
    updated_completion_summary = {
        "notebook": (
            "06_road_flood_grounding"
        )
    }

updated_completion_summary.update(
    {
        "status": "complete",
        "grounding_version": (
            GROUNDING_VERSION
        ),
        "grounding_generated_utc": (
            GROUNDING_GENERATED_UTC
        ),
        "scene_count": int(
            len(
                master_scene_profiles_updated_df
            )
        ),
        "physical_edge_record_count": int(
            len(
                master_physical_edges_updated_df
            )
        ),
        "ranking_metadata_added": True,
        "provenance_metadata_added": True,
        "reliability_labels_added": True,
        "all_final_metadata_checks_pass": bool(
            all(
                validation_results.values()
            )
        ),
        "outputs": {
            "master_scene_profiles_csv": str(
                UPDATED_MASTER_SCENE_PROFILES_CSV
            ),
            "master_scene_profiles_json": str(
                UPDATED_MASTER_SCENE_PROFILES_JSON
            ),
            "master_scene_tokens_csv": str(
                UPDATED_MASTER_SCENE_TOKENS_CSV
            ),
            "master_scene_tokens_json": str(
                UPDATED_MASTER_SCENE_TOKENS_JSON
            ),
            "master_physical_edges_csv": str(
                UPDATED_MASTER_PHYSICAL_EDGES_CSV
            ),
            "master_physical_edges_json": str(
                UPDATED_MASTER_PHYSICAL_EDGES_JSON
            ),
            "combined_grounding_csv": str(
                UPDATED_COMBINED_GROUNDING_CSV
            ),
            "combined_grounding_json": str(
                UPDATED_COMBINED_GROUNDING_JSON
            ),
            "provenance_csv": str(
                PROVENANCE_CSV
            ),
            "provenance_json": str(
                PROVENANCE_JSON
            ),
        },
    }
)

with open(
    UPDATED_COMPLETION_SUMMARY_JSON,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        updated_completion_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 14. Replace in-memory canonical variables
# ------------------------------------------------------------

master_scene_profiles_df = (
    master_scene_profiles_updated_df
)

master_scene_tokens_df = (
    master_scene_tokens_updated_df
)

master_physical_edges_df = (
    master_physical_edges_updated_df
)

combined_grounding_dataset_df = (
    combined_grounding_updated_df
)


# ------------------------------------------------------------
# 15. Final report
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("NOTEBOOK 06 FINALIZATION COMPLETE")
print("=" * 80)

print(
    f"Grounding version             : "
    f"{GROUNDING_VERSION}"
)

print(
    f"Scene profiles updated        : "
    f"{len(master_scene_profiles_df):,}"
)

print(
    f"Scene tokens updated          : "
    f"{len(master_scene_tokens_df):,}"
)

print(
    f"Physical edges updated        : "
    f"{len(master_physical_edges_df):,}"
)

print(
    f"Combined grounding records    : "
    f"{len(combined_grounding_dataset_df):,}"
)

print(
    f"Scores present                : "
    f"{master_scene_profiles_df['transportation_flood_disruption_score'].notna().sum():,}"
)

print(
    f"Ranks present                 : "
    f"{master_scene_profiles_df['transportation_flood_disruption_rank'].notna().sum():,}"
)

print(
    f"High-reliability scenes       : "
    f"{(master_scene_profiles_df['grounding_reliability'] == 'High').sum():,}"
)

print(
    f"Moderate-reliability scenes   : "
    f"{(master_scene_profiles_df['grounding_reliability'] == 'Moderate').sum():,}"
)

print(
    f"Low-reliability scenes        : "
    f"{(master_scene_profiles_df['grounding_reliability'] == 'Low').sum():,}"
)

print("\nUPDATED OUTPUT FILES")
print("-" * 80)

for output_path in [
    UPDATED_MASTER_SCENE_PROFILES_CSV,
    UPDATED_MASTER_SCENE_PROFILES_JSON,
    UPDATED_MASTER_SCENE_TOKENS_CSV,
    UPDATED_MASTER_SCENE_TOKENS_JSON,
    UPDATED_MASTER_PHYSICAL_EDGES_CSV,
    UPDATED_MASTER_PHYSICAL_EDGES_JSON,
    UPDATED_COMBINED_GROUNDING_CSV,
    UPDATED_COMBINED_GROUNDING_JSON,
    PROVENANCE_CSV,
    PROVENANCE_JSON,
    UPDATED_COMPLETION_SUMMARY_JSON,
]:
    print(output_path)


print("\nUPDATED PROFILE PREVIEW")
print("-" * 80)

display(
    master_scene_profiles_df[
        [
            "scene_id",
            "transportation_flood_disruption_score",
            "transportation_flood_disruption_rank",
            "scene_flood_burden",
            "critical_network_disruption",
            "grounding_reliability",
            "grounding_version",
            "road_buffer_meters",
            "flood_definition",
        ]
    ]
    .sort_values(
        [
            "transportation_flood_disruption_rank",
            "scene_id",
        ]
    )
    .reset_index(drop=True)
)


print("\nPROVENANCE RECORD")
print("-" * 80)

display(
    pd.DataFrame(
        [
            {
                "grounding_version": (
                    GROUNDING_VERSION
                ),
                "scene_count": len(
                    master_scene_profiles_df
                ),
                "physical_edge_count": len(
                    master_physical_edges_df
                ),
                "road_buffer_meters": (
                    ROAD_BUFFER_METERS
                ),
                "flood_definition": (
                    FLOOD_DEFINITION
                ),
                "generated_utc": (
                    GROUNDING_GENERATED_UTC
                ),
            }
        ]
    )
)

print("=" * 80)

FINALIZING NOTEBOOK 06 PROVENANCE AND RANKING METADATA



VALIDATION RESULTS
--------------------------------------------------------------------------------
profile_count_preserved                 : True
combined_count_preserved                : True
token_count_preserved                   : True
no_duplicate_profile_scenes             : True
no_duplicate_combined_scenes            : True
all_scores_present                      : True
all_ranks_present                       : True
all_versions_present                    : True
all_reliability_labels_present          : True



NOTEBOOK 06 FINALIZATION COMPLETE
Grounding version             : v1.1
Scene profiles updated        : 26
Scene tokens updated          : 26
Physical edges updated        : 6,093
Combined grounding records    : 26
Scores present                : 26
Ranks present                 : 26
High-reliability scenes       : 19
Moderate-reliability scenes   : 7
Low-reliability scenes        : 0

UPDATED OUTPUT FILES
--------------------------------------------------------------------------------
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master/master_road_flood_scene_profiles.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master/master_road_flood_scene_profiles.json
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master/master_road_flood_scene_tokens.csv
/home/adjeiowusu1/myproject/ResilientVLM/data/processed/road_flood_grounding/batch_grounding/master

,scene_id,transportation_flood_disruption_score,transportation_flood_disruption_rank,scene_flood_burden,critical_network_disruption,grounding_reliability,grounding_version,road_buffer_meters,flood_definition
0,Spain_7370579,0.7077,1,Moderate,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
1,Spain_1167260,0.3526,2,Moderate,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
2,Spain_8565131,0.3345,3,Moderate,Moderate,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
3,Somalia_970508,0.3332,4,High,Moderate,Moderate,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
4,India_500266,0.3299,5,High,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
5,India_1050276,0.3273,6,Moderate,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
6,India_956930,0.2495,7,High,High,Moderate,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
7,India_1018327,0.2164,8,High,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
8,India_383430,0.2067,9,High,High,High,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...
9,Nigeria_598959,0.1998,10,High,Moderate,Moderate,v1.1,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...



PROVENANCE RECORD
--------------------------------------------------------------------------------


,grounding_version,scene_count,physical_edge_count,road_buffer_meters,flood_definition,generated_utc
0,v1.1,26,6093,10.0,Event flood proxy = (LabelHand == 1) AND (JRCW...,2026-07-29T00:19:53+00:00
